In [ ]:
# Report the Python runtime used for the economic analysis.
import sys, time
print("HELLO NOTEBOOK", flush=True)
print("python =", sys.executable, flush=True)
time.sleep(1)
print("AFTER 1s", flush=True)


In [ ]:
# Load DuckDB for out-of-memory aggregation of mobility records.
import duckdb
print("duckdb version:", duckdb.__version__)


In [ ]:
# Aggregate park-level visits, costs, benefits, and payback outcomes.
import os
import numpy as np
import pandas as pd

CSV1 = r"data/restricted/mobility/mobility_part_0_filtered.csv"
CSV2 = r"data/restricted/mobility/mobility_part_1_filtered.csv"

COST_XLSX = r"data/restricted/park_costs/park_costs_2019_2024.xlsx"
BASE_SHP  = r"data/public/urban_parks/urban_parks_study_area.shp"

OUT_DIR = r"outputs"
os.makedirs(OUT_DIR, exist_ok=True)

DUCKDB_PATH   = os.path.join(OUT_DIR, "tmp_payback.duckdb")
PARK_PARQUET  = os.path.join(OUT_DIR, "park_agg.parquet")
USERS_TXT     = os.path.join(OUT_DIR, "approx_unique_users.txt")
OUT_TABLE_CSV = os.path.join(OUT_DIR, "park_payback_table.csv")
OUT_FIG_PNG   = os.path.join(OUT_DIR, "park_payback_map.png")

YEN_PER_STEP = 0.04
COEF_NUMERATOR = 320000
PAYBACK_HORIZON = 50.0

SIZE_MIN = 6
SIZE_MAX = 180
SIZE_Q   = 0.99

for p in [CSV1, CSV2, COST_XLSX, BASE_SHP]:
    if not os.path.exists(p):
        raise FileNotFoundError(p)

print("OK: all input paths exist")
print("OUT_DIR =", OUT_DIR)


In [ ]:
# Calculate discounted payback under the initial multi-rate specification.
import os
import time
import numpy as np
import pandas as pd

t0 = time.perf_counter()

VISIT_FILES = [
    r"data/restricted/mobility/mobility_part_0.csv",
    r"data/restricted/mobility/mobility_part_1.csv",
]

COST_XLSX = (
    r"data/restricted/park_costs/park_costs_2019_2024.xlsx"
)

OUT_DIR = (
    r"outputs"
)

OUT_TABLE_CSV = os.path.join(OUT_DIR, "park_payback_discounted_multi_rate_2019_50y.csv")
OUT_TYPE_SUMMARY_CSV = os.path.join(OUT_DIR, "park_payback_discounted_multi_rate_2019_50y_by_type.csv")
OUT_RATE_SUMMARY_CSV = os.path.join(OUT_DIR, "park_discount_rate_sensitivity_summary.csv")
OUT_DIAG_TXT = os.path.join(OUT_DIR, "park_payback_discounted_multi_rate_2019_50y_diagnostics.txt")

EVAL_YEARS = 50
DISCOUNT_RATES = [0.01, 0.02, 0.04]
MAIN_RATE = 0.04

YEN_PER_STEP_2019 = 0.04056
EXPANSION_FACTOR = 134.38
PAYBACK_HORIZON = 50

CONSTRUCTION_UNIT_COST_2014 = 12000.0

AREA_IS_HECTARE_WHEN_ONLY_AREA = False

CPI_2003_2015BASE = 97.2

CPI_2014_2015BASE = 102.8 / 103.6 * 100.0
CPI_2017_2015BASE = 100.4
CPI_2019_2015BASE = 101.8

CONSTRUCTION_TO_2019_FACTOR = CPI_2019_2015BASE / CPI_2014_2015BASE
OM_TO_2019_FACTOR = CPI_2019_2015BASE / CPI_2003_2015BASE

def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def norm_osm(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\.0$", "", regex=True)
    s = s.replace({"nan": np.nan, "None": np.nan, "": np.nan})
    return s

def first_valid(series: pd.Series):
    s = series.dropna()
    return s.iloc[0] if len(s) else np.nan

def numeric_median(series: pd.Series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    return float(s.median()) if len(s) else np.nan

def pick_column(columns, exact_candidates=(), contains_all=(), contains_any=(),
                required=True, label="column"):
    cols = list(columns)
    lower_map = {c.lower(): c for c in cols}

    for cand in exact_candidates:
        if cand in cols:
            return cand
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]

    for c in cols:
        cl = c.lower()
        ok_all = all(k.lower() in cl for k in contains_all) if contains_all else True
        ok_any = any(k.lower() in cl for k in contains_any) if contains_any else True
        if ok_all and ok_any:
            return c

    if required:
        raise KeyError(f"Cannot find {label}. Available columns:\n{cols}")
    return None

def annuity_factor(r, years):
    return np.sum(1.0 / (1.0 + r) ** np.arange(1, years + 1))

# Payback is the first year when cumulative discounted net benefits recover investment.
def discounted_payback_year_array(invest_arr, annual_net_arr, r, years):
    """
    Return the first year in which cumulative discounted annual net benefits
    recover the initial investment. Parks not recouped within the evaluation
    horizon are returned as ``np.nan``.
    """
    discount_factors = 1.0 / (1.0 + r) ** np.arange(1, years + 1)
    cum_disc = annual_net_arr[:, None] * np.cumsum(discount_factors)[None, :]
    hit = cum_disc >= invest_arr[:, None]

    out = np.full(len(invest_arr), np.nan)
    valid = (
        np.isfinite(invest_arr)
        & np.isfinite(annual_net_arr)
        & (annual_net_arr > 0)
        & (invest_arr >= 0)
    )
    any_hit = hit.any(axis=1)
    out[valid & any_hit] = hit[valid & any_hit].argmax(axis=1) + 1
    return out

frames = []
for fp in VISIT_FILES:
    if not os.path.exists(fp):
        raise FileNotFoundError(fp)
    tmp = pd.read_csv(fp, low_memory=False)
    tmp["__source_file__"] = os.path.basename(fp)
    frames.append(tmp)

visits = pd.concat(frames, ignore_index=True)
print("visit rows:", f"{len(visits):,}")

osm_col_vis = pick_column(
    visits.columns,
    exact_candidates=["osm_id"],
    contains_all=["osm", "id"],
    label="visit osm_id",
)
steps_col = pick_column(
    visits.columns,
    exact_candidates=["steps"],
    contains_any=["steps", "step"],
    label="visit steps",
)

area_m2_col_vis = None
if "area_m2" in visits.columns:
    area_m2_col_vis = "area_m2"
elif "area" in visits.columns:
    area_m2_col_vis = "area"

park_class_col = "park_class" if "park_class" in visits.columns else None
park_class_name_col = "park_class_name" if "park_class_name" in visits.columns else None

lon_col_vis = None
lat_col_vis = None
lon_candidates = [c for c in ["Lng", "lng", "lon", "Lon", "longitude", "Longitude",
                              "park_lng", "park_lon", "target_lng", "centroid_lng"] if c in visits.columns]
lat_candidates = [c for c in ["Lat", "lat", "latitude", "Latitude",
                              "park_lat", "target_lat", "centroid_lat"] if c in visits.columns]
if lon_candidates and lat_candidates:
    lon_col_vis = lon_candidates[0]
    lat_col_vis = lat_candidates[0]

visits["osm_id_norm"] = norm_osm(visits[osm_col_vis])
visits["steps_num"] = pd.to_numeric(visits[steps_col], errors="coerce").fillna(0.0)

if area_m2_col_vis is not None:
    visits["area_m2_tmp"] = pd.to_numeric(visits[area_m2_col_vis], errors="coerce")
    if area_m2_col_vis == "area" and AREA_IS_HECTARE_WHEN_ONLY_AREA:
        visits["area_m2_tmp"] = visits["area_m2_tmp"] * 10000.0
else:
    visits["area_m2_tmp"] = np.nan

agg_dict = {
    "steps_num": "sum",
    "area_m2_tmp": numeric_median,
}
rename_dict = {
    "steps_num": "steps_sum",
    "area_m2_tmp": "area_m2",
}

if park_class_col:
    agg_dict[park_class_col] = first_valid
    rename_dict[park_class_col] = "park_class"
if park_class_name_col:
    agg_dict[park_class_name_col] = first_valid
    rename_dict[park_class_name_col] = "park_class_name"
if lon_col_vis and lat_col_vis:
    agg_dict[lon_col_vis] = numeric_median
    agg_dict[lat_col_vis] = numeric_median
    rename_dict[lon_col_vis] = "Lng"
    rename_dict[lat_col_vis] = "Lat"

parks = (
    visits.dropna(subset=["osm_id_norm"])
    .groupby("osm_id_norm", as_index=False)
    .agg(agg_dict)
    .rename(columns=rename_dict)
)

print("parks after visit aggregation:", f"{len(parks):,}")

if not os.path.exists(COST_XLSX):
    raise FileNotFoundError(COST_XLSX)

cost = pd.read_excel(COST_XLSX, engine="openpyxl")
print("cost rows:", f"{len(cost):,}")

osm_col_cost = pick_column(
    cost.columns,
    exact_candidates=["osm_id"],
    contains_all=["osm", "id"],
    label="cost osm_id",
)

land_total_col = pick_column(
    cost.columns,
    exact_candidates=["land_price_1year"],
    contains_all=["land", "price", "1year"],
    required=False,
    label="land total cost column",
)
if land_total_col is None:
    raise KeyError(
        "land_price_1year is required; unit land price multiplied by area is not used as a fallback."
    )

om_unit_col = pick_column(
    cost.columns,
    exact_candidates=[
        "unit_maintenance_yen_per_m2_2024",
        "unit_maintenance_yen_per_m2",
        "annual_om_unit_yen_per_m2",
        "O&M单价",
        "养护单价",
        "维护单价",
        "管理单价",
    ],
    contains_any=["maintenance", "om", "o&m", "养护", "维护", "管理"],
    label="O&M unit cost column",
)

area_m2_col_cost = None
if "area_m2" in cost.columns:
    area_m2_col_cost = "area_m2"
elif "area" in cost.columns:
    area_m2_col_cost = "area"

cost["osm_id_norm"] = norm_osm(cost[osm_col_cost])
cost["land_price_1year_raw"] = pd.to_numeric(cost[land_total_col], errors="coerce")
cost["om_unit_cost_raw"] = pd.to_numeric(cost[om_unit_col], errors="coerce")

if area_m2_col_cost is not None:
    cost["area_m2_cost"] = pd.to_numeric(cost[area_m2_col_cost], errors="coerce")
    if area_m2_col_cost == "area" and AREA_IS_HECTARE_WHEN_ONLY_AREA:
        cost["area_m2_cost"] = cost["area_m2_cost"] * 10000.0
else:
    cost["area_m2_cost"] = np.nan

cost["land_cost_2019"] = cost["land_price_1year_raw"]

cost["om_unit_cost_2019"] = cost["om_unit_cost_raw"] * OM_TO_2019_FACTOR

cost = cost.drop_duplicates("osm_id_norm", keep="last")

df = parks.merge(
    cost[
        [
            "osm_id_norm",
            "land_cost_2019",
            "om_unit_cost_raw",
            "om_unit_cost_2019",
            "area_m2_cost",
        ]
    ],
    on="osm_id_norm",
    how="left",
)

df["area_m2"] = pd.to_numeric(df["area_m2"], errors="coerce")
df["area_m2"] = df["area_m2"].fillna(df["area_m2_cost"])

df["steps_sum"] = pd.to_numeric(df["steps_sum"], errors="coerce").fillna(0.0)

df["annual_benefit_yen_2019"] = (
    df["steps_sum"] * YEN_PER_STEP_2019 * EXPANSION_FACTOR
)

df["construction_cost_2019"] = (
    df["area_m2"] * CONSTRUCTION_UNIT_COST_2014 * CONSTRUCTION_TO_2019_FACTOR
)

df["annual_om_cost_2019"] = df["om_unit_cost_2019"] * df["area_m2"]

df["annual_net_benefit_2019"] = (
    df["annual_benefit_yen_2019"] - df["annual_om_cost_2019"]
)

df["one_time_investment_2019"] = (
    df["land_cost_2019"] + df["construction_cost_2019"]
)

df["simple_payback_years_total"] = np.where(
    (df["annual_net_benefit_2019"] > 0)
    & np.isfinite(df["annual_net_benefit_2019"])
    & np.isfinite(df["one_time_investment_2019"]),
    df["one_time_investment_2019"] / df["annual_net_benefit_2019"],
    np.nan,
)

df["simple_payback_years_land_only"] = np.where(
    (df["annual_net_benefit_2019"] > 0)
    & np.isfinite(df["annual_net_benefit_2019"])
    & np.isfinite(df["land_cost_2019"]),
    df["land_cost_2019"] / df["annual_net_benefit_2019"],
    np.nan,
)

invest_arr = pd.to_numeric(df["one_time_investment_2019"], errors="coerce").to_numpy(dtype=float)
annual_net_arr = pd.to_numeric(df["annual_net_benefit_2019"], errors="coerce").to_numpy(dtype=float)

rate_summary_rows = []

for r in DISCOUNT_RATES:
    rate_pct = int(round(r * 100))
    rate_tag = f"r{rate_pct}"

    pv_factor = annuity_factor(r, EVAL_YEARS)
    df[f"pv_factor_{rate_tag}"] = pv_factor

    df[f"pv_annual_net_benefit_50y_{rate_tag}"] = df["annual_net_benefit_2019"] * pv_factor
    df[f"npv_50y_{rate_tag}"] = (
        -df["one_time_investment_2019"] + df[f"pv_annual_net_benefit_50y_{rate_tag}"]
    )

    payback_arr = discounted_payback_year_array(
        invest_arr=invest_arr,
        annual_net_arr=annual_net_arr,
        r=r,
        years=EVAL_YEARS
    )
    df[f"discounted_payback_years_{rate_tag}"] = payback_arr
    df[f"discounted_payback_status_{rate_tag}"] = np.where(
        df[f"discounted_payback_years_{rate_tag}"].notna(),
        f"recouped within {EVAL_YEARS} years",
        f"not recouped within {EVAL_YEARS} years",
    )

    rate_summary_rows.append({
        "discount_rate": r,
        "discount_rate_pct": rate_pct,
        "pv_factor_50y": pv_factor,
        "recoup_share_50y": df[f"discounted_payback_years_{rate_tag}"].notna().mean(),
        "npv_positive_share": (df[f"npv_50y_{rate_tag}"] > 0).mean(),
        "median_discounted_payback": df[f"discounted_payback_years_{rate_tag}"].median(),
        "mean_discounted_payback": df[f"discounted_payback_years_{rate_tag}"].mean(),
    })

rate_summary = pd.DataFrame(rate_summary_rows)

MAIN_RATE_TAG = f"r{int(round(MAIN_RATE * 100))}"
df["payback_years"] = df[f"discounted_payback_years_{MAIN_RATE_TAG}"]
df["discounted_payback_status"] = df[f"discounted_payback_status_{MAIN_RATE_TAG}"]
df["npv_50y_main"] = df[f"npv_50y_{MAIN_RATE_TAG}"]

df["is_black"] = df["payback_years"].isna() | (df["payback_years"] > PAYBACK_HORIZON)
df["payback_clip"] = df["payback_years"].clip(lower=0, upper=PAYBACK_HORIZON)

type_cols = [c for c in ["park_class", "park_class_name"] if c in df.columns]

type_summary_frames = []
if type_cols:
    for r in DISCOUNT_RATES:
        rate_pct = int(round(r * 100))
        rate_tag = f"r{rate_pct}"

        summary_one = (
            df.groupby(type_cols, dropna=False, as_index=False)
            .agg(
                parks_n=("osm_id_norm", "count"),
                annual_benefit_sum_2019=("annual_benefit_yen_2019", "sum"),
                annual_om_sum_2019=("annual_om_cost_2019", "sum"),
                annual_net_sum_2019=("annual_net_benefit_2019", "sum"),
                investment_sum_2019=("one_time_investment_2019", "sum"),
                npv_50y_sum=(f"npv_50y_{rate_tag}", "sum"),
                recoup_share_50y=(f"discounted_payback_years_{rate_tag}", lambda s: s.notna().mean()),
                median_discounted_payback=(f"discounted_payback_years_{rate_tag}", "median"),
                mean_discounted_payback=(f"discounted_payback_years_{rate_tag}", "mean"),
            )
        )
        summary_one["discount_rate"] = r
        summary_one["discount_rate_pct"] = rate_pct
        type_summary_frames.append(summary_one)

if type_summary_frames:
    type_summary = pd.concat(type_summary_frames, ignore_index=True)
else:
    type_summary = pd.DataFrame()

benefit_backcheck = np.nan
mask_b = df["steps_sum"] > 0
if mask_b.any():
    benefit_backcheck = np.nanmedian(
        df.loc[mask_b, "annual_benefit_yen_2019"]
        / (df.loc[mask_b, "steps_sum"] * EXPANSION_FACTOR)
    )

match_rate_land = 1 - df["land_cost_2019"].isna().mean()
match_rate_om = 1 - df["om_unit_cost_2019"].isna().mean()
match_rate_area = 1 - df["area_m2"].isna().mean()

diag_lines = []
diag_lines.append(f"visit rows: {len(visits):,}")
diag_lines.append(f"park rows after aggregation: {len(parks):,}")
diag_lines.append(f"final park rows: {len(df):,}")
diag_lines.append("")
diag_lines.append(f"evaluation years: {EVAL_YEARS}")
diag_lines.append(f"discount rates: {DISCOUNT_RATES}")
diag_lines.append(f"main discount rate: {MAIN_RATE:.2%}")
diag_lines.append(f"yen per step (2019 JPY): {YEN_PER_STEP_2019}")
diag_lines.append(f"expansion factor: {EXPANSION_FACTOR}")
diag_lines.append("")
diag_lines.append(f"construction unit cost (2014): {CONSTRUCTION_UNIT_COST_2014:,.2f} JPY/m2")
diag_lines.append(f"construction to 2019 factor: {CONSTRUCTION_TO_2019_FACTOR:.6f}")
diag_lines.append(f"O&M to 2019 factor: {OM_TO_2019_FACTOR:.6f}")
diag_lines.append("")
diag_lines.append(f"match rate land_cost_2019: {match_rate_land:.2%}")
diag_lines.append(f"match rate om_unit_cost_2019: {match_rate_om:.2%}")
diag_lines.append(f"match rate area_m2: {match_rate_area:.2%}")
diag_lines.append("")
diag_lines.append(f"benefit back-check (should be ~{YEN_PER_STEP_2019}): {benefit_backcheck}")
diag_lines.append(f"simple payback total <=50y share: {(df['simple_payback_years_total'] <= EVAL_YEARS).mean():.2%}")
diag_lines.append(f"simple payback land only <=50y share: {(df['simple_payback_years_land_only'] <= EVAL_YEARS).mean():.2%}")

for r in DISCOUNT_RATES:
    rate_pct = int(round(r * 100))
    rate_tag = f"r{rate_pct}"
    diag_lines.append(
        f"discounted payback <=50y share @ {rate_pct}%: "
        f"{df[f'discounted_payback_years_{rate_tag}'].notna().mean():.2%}"
    )
    diag_lines.append(
        f"NPV>0 share @ {rate_pct}%: "
        f"{(df[f'npv_50y_{rate_tag}'] > 0).mean():.2%}"
    )

ensure_dir(OUT_DIR)

front_cols = [
    "osm_id_norm",
    "park_class",
    "park_class_name",
    "area_m2",
    "steps_sum",
    "annual_benefit_yen_2019",
    "om_unit_cost_raw",
    "om_unit_cost_2019",
    "annual_om_cost_2019",
    "annual_net_benefit_2019",
    "land_cost_2019",
    "construction_cost_2019",
    "one_time_investment_2019",
    "simple_payback_years_total",
    "simple_payback_years_land_only",
    "discounted_payback_years_r1",
    "discounted_payback_years_r2",
    "discounted_payback_years_r4",
    "npv_50y_r1",
    "npv_50y_r2",
    "npv_50y_r4",
    "payback_years",
    "discounted_payback_status",
    "npv_50y_main",
    "is_black",
    "payback_clip",
    "Lng",
    "Lat",
]
keep_cols = [c for c in front_cols if c in df.columns] + [c for c in df.columns if c not in front_cols]
df = df[keep_cols].copy()

df.to_csv(OUT_TABLE_CSV, index=False, encoding="utf-8-sig")
print("Saved:", OUT_TABLE_CSV)

if not type_summary.empty:
    type_summary.to_csv(OUT_TYPE_SUMMARY_CSV, index=False, encoding="utf-8-sig")
    print("Saved:", OUT_TYPE_SUMMARY_CSV)

rate_summary.to_csv(OUT_RATE_SUMMARY_CSV, index=False, encoding="utf-8-sig")
print("Saved:", OUT_RATE_SUMMARY_CSV)

with open(OUT_DIAG_TXT, "w", encoding="utf-8") as f:
    f.write("\n".join(diag_lines))
print("Saved:", OUT_DIAG_TXT)

print("\n".join(diag_lines))
print(f"Cell1 done in {time.perf_counter()-t0:.1f}s")

print("\nRate summary:")
display(rate_summary)

df.head()


In [ ]:
# Count distinct users and aggregate park-level mobility records with DuckDB.
import time
import duckdb
import os

PARK_AGG_CSV = os.path.join(OUT_DIR, "park_agg.csv")

t0 = time.perf_counter()
print("DuckDB: start aggregation (CSV + EXACT users)...", flush=True)

con = duckdb.connect(DUCKDB_PATH)

con.execute("PRAGMA threads=8;")
con.execute("PRAGMA preserve_insertion_order=false;")

con.execute(f"""
    CREATE OR REPLACE VIEW v AS
    SELECT
        CAST(user_ID AS VARCHAR) AS user_ID,
        CAST(osm_id  AS VARCHAR) AS osm_id,
        CAST(Lat     AS DOUBLE)  AS Lat,
        CAST(Lng     AS DOUBLE)  AS Lng,
        CAST(area    AS DOUBLE)  AS area,
        CAST(steps   AS DOUBLE)  AS steps,
        CAST(park_class_name AS VARCHAR) AS park_class_name
    FROM read_csv_auto('{CSV1}', ignore_errors=true)
    UNION ALL
    SELECT
        CAST(user_ID AS VARCHAR) AS user_ID,
        CAST(osm_id  AS VARCHAR) AS osm_id,
        CAST(Lat     AS DOUBLE)  AS Lat,
        CAST(Lng     AS DOUBLE)  AS Lng,
        CAST(area    AS DOUBLE)  AS area,
        CAST(steps   AS DOUBLE)  AS steps,
        CAST(park_class_name AS VARCHAR) AS park_class_name
    FROM read_csv_auto('{CSV2}', ignore_errors=true);
""")
print("DuckDB: view created", flush=True)

print("DuckDB: counting DISTINCT user_ID (EXACT)...", flush=True)
n_users_exact = con.execute("""
    SELECT COUNT(DISTINCT user_ID) AS n_users
    FROM v
    WHERE user_ID IS NOT NULL AND length(trim(user_ID)) > 0;
""").fetchone()[0]
n_users_exact = int(n_users_exact)
print(f"EXACT unique users = {n_users_exact:,}", flush=True)

with open(USERS_TXT, "w", encoding="utf-8") as f:
    f.write(str(n_users_exact))

print("DuckDB: aggregating parks -> CSV ...", flush=True)
con.execute(f"""
    COPY (
        SELECT
            osm_id,
            avg(Lat) AS Lat,
            avg(Lng) AS Lng,
            any_value(area) AS area_m2,
            any_value(park_class_name) AS park_class_name,
            sum(coalesce(steps, 0.0)) AS steps_sum,
            count(*) AS n_visits
        FROM v
        WHERE osm_id IS NOT NULL
          AND Lat IS NOT NULL AND Lng IS NOT NULL
        GROUP BY osm_id
    ) TO '{PARK_AGG_CSV}' (HEADER, DELIMITER ',');
""")

con.close()

print("Wrote users:", USERS_TXT, flush=True)
print("Wrote parks:", PARK_AGG_CSV, flush=True)
print(f"DuckDB done in {time.perf_counter()-t0:.1f}s", flush=True)


In [ ]:
# Calculate full discounted payback with a 2% main discount-rate scenario.
import os
import time
import numpy as np
import pandas as pd
from IPython.display import display

t0 = time.perf_counter()

VISIT_FILES = [
    r"data/restricted/mobility/mobility_part_0.csv",
    r"data/restricted/mobility/mobility_part_1.csv",
]

COST_XLSX = (
    r"data/restricted/park_costs/park_costs_2019_2024.xlsx"
)

OUT_DIR = (
    r"outputs"
)

OUT_TABLE_CSV = os.path.join(OUT_DIR, "park_payback_discounted_multi_rate_fullpayback_2019_50y_main2pct.csv")
OUT_TYPE_SUMMARY_CSV = os.path.join(OUT_DIR, "park_payback_discounted_multi_rate_fullpayback_2019_50y_by_type_main2pct.csv")
OUT_RATE_SUMMARY_CSV = os.path.join(OUT_DIR, "park_discount_rate_sensitivity_fullpayback_summary_main2pct.csv")
OUT_DIAG_TXT = os.path.join(OUT_DIR, "park_payback_discounted_multi_rate_fullpayback_2019_50y_diagnostics_main2pct.txt")

EVAL_YEARS = 50
DISCOUNT_RATES = [0.01, 0.02, 0.04]
MAIN_RATE = 0.02
PAYBACK_HORIZON = 50

YEN_PER_STEP_2019 = 0.04056
EXPANSION_FACTOR = 150

CONSTRUCTION_UNIT_COST_2014 = 12000.0
AREA_IS_HECTARE_WHEN_ONLY_AREA = False

FULL_PAYBACK_CAP_FOR_SUMMARY = 500.0

CPI_2003_2015BASE = 97.2
CPI_2014_2015BASE = 102.8 / 103.6 * 100.0
CPI_2019_2015BASE = 101.8

CONSTRUCTION_TO_2019_FACTOR = CPI_2019_2015BASE / CPI_2014_2015BASE
OM_TO_2019_FACTOR = CPI_2019_2015BASE / CPI_2003_2015BASE

def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def norm_osm(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\.0$", "", regex=True)
    s = s.replace({"nan": np.nan, "None": np.nan, "": np.nan})
    return s

def first_valid(series: pd.Series):
    s = series.dropna()
    return s.iloc[0] if len(s) else np.nan

def numeric_median(series: pd.Series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    return float(s.median()) if len(s) else np.nan

def pick_column(columns, exact_candidates=(), contains_all=(), contains_any=(),
                required=True, label="column"):
    cols = list(columns)
    lower_map = {c.lower(): c for c in cols}

    for cand in exact_candidates:
        if cand in cols:
            return cand
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]

    for c in cols:
        cl = c.lower()
        ok_all = all(k.lower() in cl for k in contains_all) if contains_all else True
        ok_any = any(k.lower() in cl for k in contains_any) if contains_any else True
        if ok_all and ok_any:
            return c

    if required:
        raise KeyError(f"Cannot find {label}. Available columns:\n{cols}")
    return None

def rate_tag_from_r(r: float) -> str:
    return f"r{int(round(r * 100))}"

def annuity_factor(r: float, years: int) -> float:
    if r == 0:
        return float(years)
    return float(np.sum(1.0 / (1.0 + r) ** np.arange(1, years + 1)))

def full_payback_year_array(invest_arr, annual_net_arr, r):
    invest = np.asarray(invest_arr, dtype=float)
    annual_net = np.asarray(annual_net_arr, dtype=float)

    out = np.full(len(invest), np.nan, dtype=float)
    valid_base = np.isfinite(invest) & np.isfinite(annual_net)

    mask_zero_inv = valid_base & (invest <= 0)
    out[mask_zero_inv] = 0.0

    mask_never_nonpos_net = valid_base & (invest > 0) & (annual_net <= 0)
    out[mask_never_nonpos_net] = np.inf

    mask_work = valid_base & (invest > 0) & (annual_net > 0)
    if not np.any(mask_work):
        return out

    I = invest[mask_work]
    N = annual_net[mask_work]

    if r == 0:
        out[mask_work] = np.ceil(I / N)
        return out

    mask_finite = I < (N / r)
    idx_work = np.where(mask_work)[0]

    idx_never = idx_work[~mask_finite]
    out[idx_never] = np.inf

    if np.any(mask_finite):
        I_fin = I[mask_finite]
        N_fin = N[mask_finite]
        a = r * I_fin / N_fin
        T = np.ceil(-np.log1p(-a) / np.log1p(r))
        idx_fin = idx_work[mask_finite]
        out[idx_fin] = T

    return out

def payback_within_horizon_array(full_payback_arr, years):
    arr = np.asarray(full_payback_arr, dtype=float)
    return np.where(np.isfinite(arr) & (arr <= years), arr, np.nan)

def capped_payback_stats(arr, cap=500.0):
    x = pd.to_numeric(pd.Series(arr), errors="coerce").to_numpy(dtype=float)
    x = np.where(np.isinf(x), cap, x)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan
    return float(np.median(x)), float(np.mean(x))

def finite_mean(s):
    x = pd.to_numeric(s, errors="coerce").to_numpy(dtype=float)
    x = x[np.isfinite(x)]
    return float(np.mean(x)) if len(x) else np.nan

def finite_median(s):
    x = pd.to_numeric(s, errors="coerce").to_numpy(dtype=float)
    x = x[np.isfinite(x)]
    return float(np.median(x)) if len(x) else np.nan

def capped_mean(s, cap=500.0):
    x = pd.to_numeric(s, errors="coerce").to_numpy(dtype=float)
    x = np.where(np.isinf(x), cap, x)
    x = x[np.isfinite(x)]
    return float(np.mean(x)) if len(x) else np.nan

def capped_median(s, cap=500.0):
    x = pd.to_numeric(s, errors="coerce").to_numpy(dtype=float)
    x = np.where(np.isinf(x), cap, x)
    x = x[np.isfinite(x)]
    return float(np.median(x)) if len(x) else np.nan

frames = []
for fp in VISIT_FILES:
    if not os.path.exists(fp):
        raise FileNotFoundError(fp)
    tmp = pd.read_csv(fp, low_memory=False)
    tmp["__source_file__"] = os.path.basename(fp)
    frames.append(tmp)

visits = pd.concat(frames, ignore_index=True)
print("visit rows:", f"{len(visits):,}")

osm_col_vis = pick_column(
    visits.columns,
    exact_candidates=["osm_id"],
    contains_all=["osm", "id"],
    label="visit osm_id",
)
steps_col = pick_column(
    visits.columns,
    exact_candidates=["steps"],
    contains_any=["steps", "step"],
    label="visit steps",
)

area_m2_col_vis = None
if "area_m2" in visits.columns:
    area_m2_col_vis = "area_m2"
elif "area" in visits.columns:
    area_m2_col_vis = "area"

park_class_col = "park_class" if "park_class" in visits.columns else None
park_class_name_col = "park_class_name" if "park_class_name" in visits.columns else None

lon_col_vis = None
lat_col_vis = None
lon_candidates = [c for c in ["Lng", "lng", "lon", "Lon", "longitude", "Longitude",
                              "park_lng", "park_lon", "target_lng", "centroid_lng"] if c in visits.columns]
lat_candidates = [c for c in ["Lat", "lat", "latitude", "Latitude",
                              "park_lat", "target_lat", "centroid_lat"] if c in visits.columns]
if lon_candidates and lat_candidates:
    lon_col_vis = lon_candidates[0]
    lat_col_vis = lat_candidates[0]

visits["osm_id_norm"] = norm_osm(visits[osm_col_vis])
visits["steps_num"] = pd.to_numeric(visits[steps_col], errors="coerce").fillna(0.0)

if area_m2_col_vis is not None:
    visits["area_m2_tmp"] = pd.to_numeric(visits[area_m2_col_vis], errors="coerce")
    if area_m2_col_vis == "area" and AREA_IS_HECTARE_WHEN_ONLY_AREA:
        visits["area_m2_tmp"] = visits["area_m2_tmp"] * 10000.0
else:
    visits["area_m2_tmp"] = np.nan

agg_dict = {
    "steps_num": "sum",
    "area_m2_tmp": numeric_median,
}
rename_dict = {
    "steps_num": "steps_sum",
    "area_m2_tmp": "area_m2",
}

if park_class_col:
    agg_dict[park_class_col] = first_valid
    rename_dict[park_class_col] = "park_class"
if park_class_name_col:
    agg_dict[park_class_name_col] = first_valid
    rename_dict[park_class_name_col] = "park_class_name"
if lon_col_vis and lat_col_vis:
    agg_dict[lon_col_vis] = numeric_median
    agg_dict[lat_col_vis] = numeric_median
    rename_dict[lon_col_vis] = "Lng"
    rename_dict[lat_col_vis] = "Lat"

parks = (
    visits.dropna(subset=["osm_id_norm"])
    .groupby("osm_id_norm", as_index=False)
    .agg(agg_dict)
    .rename(columns=rename_dict)
)
print("parks after visit aggregation:", f"{len(parks):,}")

if not os.path.exists(COST_XLSX):
    raise FileNotFoundError(COST_XLSX)

cost = pd.read_excel(COST_XLSX, engine="openpyxl")
print("cost rows:", f"{len(cost):,}")

osm_col_cost = pick_column(
    cost.columns,
    exact_candidates=["osm_id"],
    contains_all=["osm", "id"],
    label="cost osm_id",
)

land_total_col = pick_column(
    cost.columns,
    exact_candidates=["land_price_1year"],
    contains_all=["land", "price", "1year"],
    required=False,
    label="land total cost column",
)
if land_total_col is None:
    raise KeyError("land_price_1year was not found.")

om_unit_col = pick_column(
    cost.columns,
    exact_candidates=[
        "unit_maintenance_yen_per_m2_2024",
        "unit_maintenance_yen_per_m2",
        "annual_om_unit_yen_per_m2",
        "O&M单价",
        "养护单价",
        "维护单价",
        "管理单价",
    ],
    contains_any=["maintenance", "om", "o&m", "养护", "维护", "管理"],
    label="O&M unit cost column",
)

area_m2_col_cost = None
if "area_m2" in cost.columns:
    area_m2_col_cost = "area_m2"
elif "area" in cost.columns:
    area_m2_col_cost = "area"

cost["osm_id_norm"] = norm_osm(cost[osm_col_cost])
cost["land_price_1year_raw"] = pd.to_numeric(cost[land_total_col], errors="coerce")
cost["om_unit_cost_raw"] = pd.to_numeric(cost[om_unit_col], errors="coerce")

if area_m2_col_cost is not None:
    cost["area_m2_cost"] = pd.to_numeric(cost[area_m2_col_cost], errors="coerce")
    if area_m2_col_cost == "area" and AREA_IS_HECTARE_WHEN_ONLY_AREA:
        cost["area_m2_cost"] = cost["area_m2_cost"] * 10000.0
else:
    cost["area_m2_cost"] = np.nan

cost["land_cost_2019"] = cost["land_price_1year_raw"]
cost["om_unit_cost_2019"] = cost["om_unit_cost_raw"] * OM_TO_2019_FACTOR
cost = cost.drop_duplicates("osm_id_norm", keep="last")

df = parks.merge(
    cost[
        [
            "osm_id_norm",
            "land_cost_2019",
            "om_unit_cost_raw",
            "om_unit_cost_2019",
            "area_m2_cost",
        ]
    ],
    on="osm_id_norm",
    how="left",
)

df["area_m2"] = pd.to_numeric(df["area_m2"], errors="coerce")
df["area_m2"] = df["area_m2"].fillna(df["area_m2_cost"])

df["steps_sum"] = pd.to_numeric(df["steps_sum"], errors="coerce").fillna(0.0)

df["annual_benefit_yen_2019"] = df["steps_sum"] * YEN_PER_STEP_2019 * EXPANSION_FACTOR
df["construction_cost_2019"] = df["area_m2"] * CONSTRUCTION_UNIT_COST_2014 * CONSTRUCTION_TO_2019_FACTOR
df["annual_om_cost_2019"] = df["om_unit_cost_2019"] * df["area_m2"]
df["annual_net_benefit_2019"] = df["annual_benefit_yen_2019"] - df["annual_om_cost_2019"]
df["one_time_investment_2019"] = df["land_cost_2019"] + df["construction_cost_2019"]

df["simple_payback_years_total"] = np.where(
    (df["annual_net_benefit_2019"] > 0)
    & np.isfinite(df["annual_net_benefit_2019"])
    & np.isfinite(df["one_time_investment_2019"]),
    df["one_time_investment_2019"] / df["annual_net_benefit_2019"],
    np.nan,
)

df["simple_payback_years_land_only"] = np.where(
    (df["annual_net_benefit_2019"] > 0)
    & np.isfinite(df["annual_net_benefit_2019"])
    & np.isfinite(df["land_cost_2019"]),
    df["land_cost_2019"] / df["annual_net_benefit_2019"],
    np.nan,
)

invest_arr = pd.to_numeric(df["one_time_investment_2019"], errors="coerce").to_numpy(dtype=float)
annual_net_arr = pd.to_numeric(df["annual_net_benefit_2019"], errors="coerce").to_numpy(dtype=float)

rate_summary_rows = []

for r in DISCOUNT_RATES:
    rate_tag = rate_tag_from_r(r)
    rate_pct = int(round(r * 100))

    pv_factor = annuity_factor(r, EVAL_YEARS)
    df[f"pv_factor_50y_{rate_tag}"] = pv_factor

    df[f"pv_annual_net_benefit_50y_{rate_tag}"] = df["annual_net_benefit_2019"] * pv_factor
    df[f"npv_50y_{rate_tag}"] = -df["one_time_investment_2019"] + df[f"pv_annual_net_benefit_50y_{rate_tag}"]

    full_payback_arr = full_payback_year_array(
        invest_arr=invest_arr,
        annual_net_arr=annual_net_arr,
        r=r
    )
    df[f"discounted_payback_years_{rate_tag}"] = full_payback_arr

    within_50_arr = payback_within_horizon_array(full_payback_arr, EVAL_YEARS)
    df[f"discounted_payback_years_50y_{rate_tag}"] = within_50_arr
    df[f"recoup_within_50y_{rate_tag}"] = np.isfinite(within_50_arr)

    full_status = np.full(len(df), "invalid", dtype=object)
    full_status[np.isfinite(full_payback_arr)] = "finite payback"
    full_status[np.isinf(full_payback_arr)] = "never pay back"
    df[f"discounted_payback_status_full_{rate_tag}"] = full_status

    status_50y = np.full(len(df), f"not recouped within {EVAL_YEARS} years", dtype=object)
    status_50y[np.isfinite(within_50_arr)] = f"recouped within {EVAL_YEARS} years"
    status_50y[np.isinf(full_payback_arr)] = "never pay back"
    df[f"discounted_payback_status_50y_{rate_tag}"] = status_50y

    finite_full = pd.Series(full_payback_arr)[np.isfinite(full_payback_arr)]
    median_capped, mean_capped = capped_payback_stats(full_payback_arr, cap=FULL_PAYBACK_CAP_FOR_SUMMARY)

    rate_summary_rows.append({
        "discount_rate": r,
        "discount_rate_pct": rate_pct,
        "pv_factor_50y": pv_factor,
        "recoup_share_50y": float(np.isfinite(within_50_arr).mean()),
        "npv_positive_share": float((df[f"npv_50y_{rate_tag}"] > 0).mean()),
        "never_payback_share": float(np.isinf(full_payback_arr).mean()),
        "median_full_payback_years_finite_only": float(finite_full.median()) if len(finite_full) else np.nan,
        "mean_full_payback_years_finite_only": float(finite_full.mean()) if len(finite_full) else np.nan,
        f"median_full_payback_years_capped{int(FULL_PAYBACK_CAP_FOR_SUMMARY)}": median_capped,
        f"mean_full_payback_years_capped{int(FULL_PAYBACK_CAP_FOR_SUMMARY)}": mean_capped,
        "median_payback_within_50y": float(pd.Series(within_50_arr).median()) if np.isfinite(within_50_arr).any() else np.nan,
        "mean_payback_within_50y": float(pd.Series(within_50_arr).mean()) if np.isfinite(within_50_arr).any() else np.nan,
    })

rate_summary = pd.DataFrame(rate_summary_rows)

MAIN_RATE_TAG = rate_tag_from_r(MAIN_RATE)

df["payback_years"] = df[f"discounted_payback_years_{MAIN_RATE_TAG}"]
df["payback_years_50y"] = df[f"discounted_payback_years_50y_{MAIN_RATE_TAG}"]
df["discounted_payback_status"] = df[f"discounted_payback_status_50y_{MAIN_RATE_TAG}"]
df["npv_50y_main"] = df[f"npv_50y_{MAIN_RATE_TAG}"]

df["is_black"] = (~np.isfinite(df["payback_years"])) | (df["payback_years"] > PAYBACK_HORIZON)

df["payback_clip"] = pd.to_numeric(df["payback_years"], errors="coerce")
df.loc[~np.isfinite(df["payback_clip"]), "payback_clip"] = PAYBACK_HORIZON
df["payback_clip"] = df["payback_clip"].clip(lower=0, upper=PAYBACK_HORIZON)

type_cols = [c for c in ["park_class", "park_class_name"] if c in df.columns]

type_summary_frames = []
if type_cols:
    for r in DISCOUNT_RATES:
        rate_tag = rate_tag_from_r(r)
        rate_pct = int(round(r * 100))

        summary_one = (
            df.groupby(type_cols, dropna=False, as_index=False)
            .agg(
                parks_n=("osm_id_norm", "count"),
                annual_benefit_sum_2019=("annual_benefit_yen_2019", "sum"),
                annual_om_sum_2019=("annual_om_cost_2019", "sum"),
                annual_net_sum_2019=("annual_net_benefit_2019", "sum"),
                investment_sum_2019=("one_time_investment_2019", "sum"),
                npv_50y_sum=(f"npv_50y_{rate_tag}", "sum"),
                recoup_share_50y=(f"recoup_within_50y_{rate_tag}", "mean"),
                never_payback_share=(f"discounted_payback_years_{rate_tag}", lambda s: np.isinf(pd.to_numeric(s, errors='coerce')).mean()),
                median_full_payback_years_finite_only=(f"discounted_payback_years_{rate_tag}", finite_median),
                mean_full_payback_years_finite_only=(f"discounted_payback_years_{rate_tag}", finite_mean),
                median_full_payback_years_capped500=(f"discounted_payback_years_{rate_tag}", lambda s: capped_median(s, cap=FULL_PAYBACK_CAP_FOR_SUMMARY)),
                mean_full_payback_years_capped500=(f"discounted_payback_years_{rate_tag}", lambda s: capped_mean(s, cap=FULL_PAYBACK_CAP_FOR_SUMMARY)),
                median_payback_within_50y=(f"discounted_payback_years_50y_{rate_tag}", "median"),
                mean_payback_within_50y=(f"discounted_payback_years_50y_{rate_tag}", "mean"),
            )
        )
        summary_one["discount_rate"] = r
        summary_one["discount_rate_pct"] = rate_pct
        type_summary_frames.append(summary_one)

type_summary = pd.concat(type_summary_frames, ignore_index=True) if type_summary_frames else pd.DataFrame()

benefit_backcheck = np.nan
mask_b = df["steps_sum"] > 0
if mask_b.any():
    benefit_backcheck = np.nanmedian(
        df.loc[mask_b, "annual_benefit_yen_2019"] /
        (df.loc[mask_b, "steps_sum"] * EXPANSION_FACTOR)
    )

match_rate_land = 1 - df["land_cost_2019"].isna().mean()
match_rate_om = 1 - df["om_unit_cost_2019"].isna().mean()
match_rate_area = 1 - df["area_m2"].isna().mean()

diag_lines = []
diag_lines.append(f"visit rows: {len(visits):,}")
diag_lines.append(f"park rows after visit aggregation: {len(parks):,}")
diag_lines.append(f"final park rows: {len(df):,}")
diag_lines.append("")
diag_lines.append(f"evaluation years: {EVAL_YEARS}")
diag_lines.append(f"discount rates: {DISCOUNT_RATES}")
diag_lines.append(f"main discount rate: {MAIN_RATE:.2%}")
diag_lines.append(f"yen per step (2019 JPY): {YEN_PER_STEP_2019}")
diag_lines.append(f"expansion factor: {EXPANSION_FACTOR}")
diag_lines.append("")
diag_lines.append(f"construction unit cost (2014): {CONSTRUCTION_UNIT_COST_2014:,.2f} JPY/m2")
diag_lines.append(f"construction to 2019 factor: {CONSTRUCTION_TO_2019_FACTOR:.6f}")
diag_lines.append(f"O&M to 2019 factor: {OM_TO_2019_FACTOR:.6f}")
diag_lines.append("")
diag_lines.append(f"match rate land_cost_2019: {match_rate_land:.2%}")
diag_lines.append(f"match rate om_unit_cost_2019: {match_rate_om:.2%}")
diag_lines.append(f"match rate area_m2: {match_rate_area:.2%}")
diag_lines.append("")
diag_lines.append(f"benefit back-check (should be ~{YEN_PER_STEP_2019}): {benefit_backcheck}")
diag_lines.append(f"simple payback total <=50y share: {(df['simple_payback_years_total'] <= EVAL_YEARS).mean():.2%}")
diag_lines.append(f"simple payback land only <=50y share: {(df['simple_payback_years_land_only'] <= EVAL_YEARS).mean():.2%}")
diag_lines.append("")

for r in DISCOUNT_RATES:
    rate_tag = rate_tag_from_r(r)
    rate_pct = int(round(r * 100))
    diag_lines.append(f"----- rate = {rate_pct}% -----")
    diag_lines.append(f"recoup within 50y share @ {rate_pct}%: {df[f'recoup_within_50y_{rate_tag}'].mean():.2%}")
    diag_lines.append(f"NPV>0 share @ {rate_pct}%: {(df[f'npv_50y_{rate_tag}'] > 0).mean():.2%}")
    diag_lines.append(f"never payback share @ {rate_pct}%: {np.isinf(pd.to_numeric(df[f'discounted_payback_years_{rate_tag}'], errors='coerce')).mean():.2%}")

ensure_dir(OUT_DIR)

front_cols = [
    "osm_id_norm",
    "park_class",
    "park_class_name",
    "area_m2",
    "steps_sum",
    "annual_benefit_yen_2019",
    "om_unit_cost_raw",
    "om_unit_cost_2019",
    "annual_om_cost_2019",
    "annual_net_benefit_2019",
    "land_cost_2019",
    "construction_cost_2019",
    "one_time_investment_2019",
    "simple_payback_years_total",
    "simple_payback_years_land_only",
    "discounted_payback_years_r1",
    "discounted_payback_years_r2",
    "discounted_payback_years_r4",
    "discounted_payback_years_50y_r1",
    "discounted_payback_years_50y_r2",
    "discounted_payback_years_50y_r4",
    "npv_50y_r1",
    "npv_50y_r2",
    "npv_50y_r4",
    "payback_years",
    "payback_years_50y",
    "discounted_payback_status",
    "npv_50y_main",
    "is_black",
    "payback_clip",
    "Lng",
    "Lat",
]
keep_cols = [c for c in front_cols if c in df.columns] + [c for c in df.columns if c not in front_cols]
df = df[keep_cols].copy()

df.to_csv(OUT_TABLE_CSV, index=False, encoding="utf-8-sig")
print("Saved:", OUT_TABLE_CSV)

if not type_summary.empty:
    type_summary.to_csv(OUT_TYPE_SUMMARY_CSV, index=False, encoding="utf-8-sig")
    print("Saved:", OUT_TYPE_SUMMARY_CSV)

rate_summary.to_csv(OUT_RATE_SUMMARY_CSV, index=False, encoding="utf-8-sig")
print("Saved:", OUT_RATE_SUMMARY_CSV)

with open(OUT_DIAG_TXT, "w", encoding="utf-8") as f:
    f.write("\n".join(diag_lines))
print("Saved:", OUT_DIAG_TXT)

print("\n".join(diag_lines))
print(f"\nDone in {time.perf_counter() - t0:.1f}s")

print("\nRate summary:")
display(rate_summary)

print("\nPreview of output table:")
display(df.head())


In [ ]:
# Calculate full discounted payback across zero and positive discount rates.
import os
import time
import numpy as np
import pandas as pd

t0 = time.perf_counter()

VISIT_FILES = [
    r"data/restricted/mobility/mobility_part_0.csv",
    r"data/restricted/mobility/mobility_part_1.csv",
]

COST_XLSX = (
    r"data/restricted/park_costs/park_costs_2019_2024.xlsx"
)

OUT_DIR = (
    r"outputs"
)

OUT_TABLE_CSV = os.path.join(OUT_DIR, "park_payback_discounted_multi_rate_fullpayback_2019_50y.csv")
OUT_TYPE_SUMMARY_CSV = os.path.join(OUT_DIR, "park_payback_discounted_multi_rate_fullpayback_2019_50y_by_type.csv")
OUT_RATE_SUMMARY_CSV = os.path.join(OUT_DIR, "park_discount_rate_sensitivity_fullpayback_summary.csv")
OUT_DIAG_TXT = os.path.join(OUT_DIR, "park_payback_discounted_multi_rate_fullpayback_2019_50y_diagnostics.txt")

EVAL_YEARS = 50
DISCOUNT_RATES = [0.00, 0.01, 0.02, 0.04]
MAIN_RATE = 0.04
PAYBACK_HORIZON = 50

YEN_PER_STEP_2019 = 0.04056
EXPANSION_FACTOR = 150

CONSTRUCTION_UNIT_COST_2014 = 12000.0

AREA_IS_HECTARE_WHEN_ONLY_AREA = False

FULL_PAYBACK_CAP_FOR_SUMMARY = 500.0

CPI_2003_2015BASE = 97.2
CPI_2014_2015BASE = 102.8 / 103.6 * 100.0
CPI_2017_2015BASE = 100.4
CPI_2019_2015BASE = 101.8

CONSTRUCTION_TO_2019_FACTOR = CPI_2019_2015BASE / CPI_2014_2015BASE
OM_TO_2019_FACTOR = CPI_2019_2015BASE / CPI_2003_2015BASE

def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def norm_osm(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\.0$", "", regex=True)
    s = s.replace({"nan": np.nan, "None": np.nan, "": np.nan})
    return s

def first_valid(series: pd.Series):
    s = series.dropna()
    return s.iloc[0] if len(s) else np.nan

def numeric_median(series: pd.Series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    return float(s.median()) if len(s) else np.nan

def pick_column(columns, exact_candidates=(), contains_all=(), contains_any=(),
                required=True, label="column"):
    cols = list(columns)
    lower_map = {c.lower(): c for c in cols}

    for cand in exact_candidates:
        if cand in cols:
            return cand
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]

    for c in cols:
        cl = c.lower()
        ok_all = all(k.lower() in cl for k in contains_all) if contains_all else True
        ok_any = any(k.lower() in cl for k in contains_any) if contains_any else True
        if ok_all and ok_any:
            return c

    if required:
        raise KeyError(f"Cannot find {label}. Available columns:\n{cols}")
    return None

def rate_tag_from_r(r: float) -> str:
    return f"r{int(round(r * 100))}"

def annuity_factor(r: float, years: int) -> float:
    """
    Return the present-value annuity factor for the evaluation horizon.
    At a zero discount rate, the factor equals the number of years.
    """
    if r == 0:
        return float(years)
    return float(np.sum(1.0 / (1.0 + r) ** np.arange(1, years + 1)))

def full_payback_year_array(invest_arr, annual_net_arr, r):
    """
    Return full discounted payback without imposing the 50-year horizon.

    Finite values are the earliest whole payback year, ``np.inf`` denotes
    non-recoverable investment, and ``np.nan`` denotes invalid input. At a
    positive discount rate, constant annual net benefits have the limiting
    present value ``annual_net / r``.
    """
    invest = np.asarray(invest_arr, dtype=float)
    annual_net = np.asarray(annual_net_arr, dtype=float)

    out = np.full(len(invest), np.nan, dtype=float)

    valid_base = np.isfinite(invest) & np.isfinite(annual_net)

    mask_zero_inv = valid_base & (invest <= 0)
    out[mask_zero_inv] = 0.0

    mask_never_nonpos_net = valid_base & (invest > 0) & (annual_net <= 0)
    out[mask_never_nonpos_net] = np.inf

    mask_work = valid_base & (invest > 0) & (annual_net > 0)

    if not np.any(mask_work):
        return out

    I = invest[mask_work]
    N = annual_net[mask_work]

    if r == 0:
        out[mask_work] = np.ceil(I / N)
        return out

    # N/r is the limiting present value of a perpetual constant annual net benefit.
    mask_finite = I < (N / r)
    idx_work = np.where(mask_work)[0]

    idx_never = idx_work[~mask_finite]
    out[idx_never] = np.inf

    if np.any(mask_finite):
        I_fin = I[mask_finite]
        N_fin = N[mask_finite]
        a = r * I_fin / N_fin
        T = np.ceil(-np.log1p(-a) / np.log1p(r))
        idx_fin = idx_work[mask_finite]
        out[idx_fin] = T

    return out

def payback_within_horizon_array(full_payback_arr, years):
    """
    Retain payback years within the evaluation horizon and encode later or
    non-recoverable outcomes as ``np.nan``.
    """
    arr = np.asarray(full_payback_arr, dtype=float)
    out = np.where(np.isfinite(arr) & (arr <= years), arr, np.nan)
    return out

def capped_payback_stats(arr, cap=500.0):
    """
    Cap infinite payback values only when computing descriptive statistics;
    the underlying park-level estimates remain unchanged.
    """
    x = pd.to_numeric(pd.Series(arr), errors="coerce").to_numpy(dtype=float)
    x = np.where(np.isinf(x), cap, x)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan
    return float(np.median(x)), float(np.mean(x))

def finite_mean(s):
    x = pd.to_numeric(s, errors="coerce").to_numpy(dtype=float)
    x = x[np.isfinite(x)]
    return float(np.mean(x)) if len(x) else np.nan

def finite_median(s):
    x = pd.to_numeric(s, errors="coerce").to_numpy(dtype=float)
    x = x[np.isfinite(x)]
    return float(np.median(x)) if len(x) else np.nan

def capped_mean(s, cap=500.0):
    x = pd.to_numeric(s, errors="coerce").to_numpy(dtype=float)
    x = np.where(np.isinf(x), cap, x)
    x = x[np.isfinite(x)]
    return float(np.mean(x)) if len(x) else np.nan

def capped_median(s, cap=500.0):
    x = pd.to_numeric(s, errors="coerce").to_numpy(dtype=float)
    x = np.where(np.isinf(x), cap, x)
    x = x[np.isfinite(x)]
    return float(np.median(x)) if len(x) else np.nan

frames = []
for fp in VISIT_FILES:
    if not os.path.exists(fp):
        raise FileNotFoundError(fp)
    tmp = pd.read_csv(fp, low_memory=False)
    tmp["__source_file__"] = os.path.basename(fp)
    frames.append(tmp)

visits = pd.concat(frames, ignore_index=True)
print("visit rows:", f"{len(visits):,}")

osm_col_vis = pick_column(
    visits.columns,
    exact_candidates=["osm_id"],
    contains_all=["osm", "id"],
    label="visit osm_id",
)
steps_col = pick_column(
    visits.columns,
    exact_candidates=["steps"],
    contains_any=["steps", "step"],
    label="visit steps",
)

area_m2_col_vis = None
if "area_m2" in visits.columns:
    area_m2_col_vis = "area_m2"
elif "area" in visits.columns:
    area_m2_col_vis = "area"

park_class_col = "park_class" if "park_class" in visits.columns else None
park_class_name_col = "park_class_name" if "park_class_name" in visits.columns else None

lon_col_vis = None
lat_col_vis = None
lon_candidates = [c for c in ["Lng", "lng", "lon", "Lon", "longitude", "Longitude",
                              "park_lng", "park_lon", "target_lng", "centroid_lng"] if c in visits.columns]
lat_candidates = [c for c in ["Lat", "lat", "latitude", "Latitude",
                              "park_lat", "target_lat", "centroid_lat"] if c in visits.columns]
if lon_candidates and lat_candidates:
    lon_col_vis = lon_candidates[0]
    lat_col_vis = lat_candidates[0]

visits["osm_id_norm"] = norm_osm(visits[osm_col_vis])
visits["steps_num"] = pd.to_numeric(visits[steps_col], errors="coerce").fillna(0.0)

if area_m2_col_vis is not None:
    visits["area_m2_tmp"] = pd.to_numeric(visits[area_m2_col_vis], errors="coerce")
    if area_m2_col_vis == "area" and AREA_IS_HECTARE_WHEN_ONLY_AREA:
        visits["area_m2_tmp"] = visits["area_m2_tmp"] * 10000.0
else:
    visits["area_m2_tmp"] = np.nan

agg_dict = {
    "steps_num": "sum",
    "area_m2_tmp": numeric_median,
}
rename_dict = {
    "steps_num": "steps_sum",
    "area_m2_tmp": "area_m2",
}

if park_class_col:
    agg_dict[park_class_col] = first_valid
    rename_dict[park_class_col] = "park_class"
if park_class_name_col:
    agg_dict[park_class_name_col] = first_valid
    rename_dict[park_class_name_col] = "park_class_name"
if lon_col_vis and lat_col_vis:
    agg_dict[lon_col_vis] = numeric_median
    agg_dict[lat_col_vis] = numeric_median
    rename_dict[lon_col_vis] = "Lng"
    rename_dict[lat_col_vis] = "Lat"

parks = (
    visits.dropna(subset=["osm_id_norm"])
    .groupby("osm_id_norm", as_index=False)
    .agg(agg_dict)
    .rename(columns=rename_dict)
)

print("parks after visit aggregation:", f"{len(parks):,}")

if not os.path.exists(COST_XLSX):
    raise FileNotFoundError(COST_XLSX)

cost = pd.read_excel(COST_XLSX, engine="openpyxl")
print("cost rows:", f"{len(cost):,}")

osm_col_cost = pick_column(
    cost.columns,
    exact_candidates=["osm_id"],
    contains_all=["osm", "id"],
    label="cost osm_id",
)

land_total_col = pick_column(
    cost.columns,
    exact_candidates=["land_price_1year"],
    contains_all=["land", "price", "1year"],
    required=False,
    label="land total cost column",
)
if land_total_col is None:
    raise KeyError("land_price_1year is required; unit land price multiplied by area is not used as a fallback.")

om_unit_col = pick_column(
    cost.columns,
    exact_candidates=[
        "unit_maintenance_yen_per_m2_2024",
        "unit_maintenance_yen_per_m2",
        "annual_om_unit_yen_per_m2",
        "O&M单价",
        "养护单价",
        "维护单价",
        "管理单价",
    ],
    contains_any=["maintenance", "om", "o&m", "养护", "维护", "管理"],
    label="O&M unit cost column",
)

area_m2_col_cost = None
if "area_m2" in cost.columns:
    area_m2_col_cost = "area_m2"
elif "area" in cost.columns:
    area_m2_col_cost = "area"

cost["osm_id_norm"] = norm_osm(cost[osm_col_cost])
cost["land_price_1year_raw"] = pd.to_numeric(cost[land_total_col], errors="coerce")
cost["om_unit_cost_raw"] = pd.to_numeric(cost[om_unit_col], errors="coerce")

if area_m2_col_cost is not None:
    cost["area_m2_cost"] = pd.to_numeric(cost[area_m2_col_cost], errors="coerce")
    if area_m2_col_cost == "area" and AREA_IS_HECTARE_WHEN_ONLY_AREA:
        cost["area_m2_cost"] = cost["area_m2_cost"] * 10000.0
else:
    cost["area_m2_cost"] = np.nan

cost["land_cost_2019"] = cost["land_price_1year_raw"]

cost["om_unit_cost_2019"] = cost["om_unit_cost_raw"] * OM_TO_2019_FACTOR

cost = cost.drop_duplicates("osm_id_norm", keep="last")

df = parks.merge(
    cost[
        [
            "osm_id_norm",
            "land_cost_2019",
            "om_unit_cost_raw",
            "om_unit_cost_2019",
            "area_m2_cost",
        ]
    ],
    on="osm_id_norm",
    how="left",
)

df["area_m2"] = pd.to_numeric(df["area_m2"], errors="coerce")
df["area_m2"] = df["area_m2"].fillna(df["area_m2_cost"])

df["steps_sum"] = pd.to_numeric(df["steps_sum"], errors="coerce").fillna(0.0)

df["annual_benefit_yen_2019"] = (
    df["steps_sum"] * YEN_PER_STEP_2019 * EXPANSION_FACTOR
)

df["construction_cost_2019"] = (
    df["area_m2"] * CONSTRUCTION_UNIT_COST_2014 * CONSTRUCTION_TO_2019_FACTOR
)

df["annual_om_cost_2019"] = df["om_unit_cost_2019"] * df["area_m2"]

df["annual_net_benefit_2019"] = (
    df["annual_benefit_yen_2019"] - df["annual_om_cost_2019"]
)

df["one_time_investment_2019"] = (
    df["land_cost_2019"] + df["construction_cost_2019"]
)

df["simple_payback_years_total"] = np.where(
    (df["annual_net_benefit_2019"] > 0)
    & np.isfinite(df["annual_net_benefit_2019"])
    & np.isfinite(df["one_time_investment_2019"]),
    df["one_time_investment_2019"] / df["annual_net_benefit_2019"],
    np.nan,
)

df["simple_payback_years_land_only"] = np.where(
    (df["annual_net_benefit_2019"] > 0)
    & np.isfinite(df["annual_net_benefit_2019"])
    & np.isfinite(df["land_cost_2019"]),
    df["land_cost_2019"] / df["annual_net_benefit_2019"],
    np.nan,
)

invest_arr = pd.to_numeric(df["one_time_investment_2019"], errors="coerce").to_numpy(dtype=float)
annual_net_arr = pd.to_numeric(df["annual_net_benefit_2019"], errors="coerce").to_numpy(dtype=float)

rate_summary_rows = []

for r in DISCOUNT_RATES:
    rate_tag = rate_tag_from_r(r)
    rate_pct = int(round(r * 100))

    pv_factor = annuity_factor(r, EVAL_YEARS)
    df[f"pv_factor_50y_{rate_tag}"] = pv_factor

    df[f"pv_annual_net_benefit_50y_{rate_tag}"] = df["annual_net_benefit_2019"] * pv_factor
    df[f"npv_50y_{rate_tag}"] = (
        -df["one_time_investment_2019"] + df[f"pv_annual_net_benefit_50y_{rate_tag}"]
    )

    full_payback_arr = full_payback_year_array(
        invest_arr=invest_arr,
        annual_net_arr=annual_net_arr,
        r=r
    )
    df[f"discounted_payback_years_{rate_tag}"] = full_payback_arr

    within_50_arr = payback_within_horizon_array(full_payback_arr, EVAL_YEARS)
    df[f"discounted_payback_years_50y_{rate_tag}"] = within_50_arr
    df[f"recoup_within_50y_{rate_tag}"] = np.isfinite(within_50_arr)

    full_status = np.full(len(df), "invalid", dtype=object)
    full_status[np.isfinite(full_payback_arr)] = "finite payback"
    full_status[np.isinf(full_payback_arr)] = "never pay back"
    df[f"discounted_payback_status_full_{rate_tag}"] = full_status

    status_50y = np.full(len(df), f"not recouped within {EVAL_YEARS} years", dtype=object)
    status_50y[np.isfinite(within_50_arr)] = f"recouped within {EVAL_YEARS} years"
    status_50y[np.isinf(full_payback_arr)] = "never pay back"
    df[f"discounted_payback_status_50y_{rate_tag}"] = status_50y

    full_payback_series = pd.Series(full_payback_arr)
    finite_full = full_payback_series[np.isfinite(full_payback_series)]

    median_capped, mean_capped = capped_payback_stats(
        full_payback_arr,
        cap=FULL_PAYBACK_CAP_FOR_SUMMARY
    )

    rate_summary_rows.append({
        "discount_rate": r,
        "discount_rate_pct": rate_pct,
        "pv_factor_50y": pv_factor,
        "recoup_share_50y": float(np.isfinite(within_50_arr).mean()),
        "npv_positive_share": float((df[f"npv_50y_{rate_tag}"] > 0).mean()),
        "never_payback_share": float(np.isinf(full_payback_arr).mean()),

        "median_full_payback_years_finite_only": float(finite_full.median()) if len(finite_full) else np.nan,
        "mean_full_payback_years_finite_only": float(finite_full.mean()) if len(finite_full) else np.nan,

        f"median_full_payback_years_capped{int(FULL_PAYBACK_CAP_FOR_SUMMARY)}": median_capped,
        f"mean_full_payback_years_capped{int(FULL_PAYBACK_CAP_FOR_SUMMARY)}": mean_capped,

        "median_payback_within_50y": float(pd.Series(within_50_arr).median()) if np.isfinite(within_50_arr).any() else np.nan,
        "mean_payback_within_50y": float(pd.Series(within_50_arr).mean()) if np.isfinite(within_50_arr).any() else np.nan,
    })

rate_summary = pd.DataFrame(rate_summary_rows)

MAIN_RATE_TAG = rate_tag_from_r(MAIN_RATE)

df["payback_years"] = df[f"discounted_payback_years_{MAIN_RATE_TAG}"]
df["payback_years_50y"] = df[f"discounted_payback_years_50y_{MAIN_RATE_TAG}"]
df["discounted_payback_status"] = df[f"discounted_payback_status_50y_{MAIN_RATE_TAG}"]
df["npv_50y_main"] = df[f"npv_50y_{MAIN_RATE_TAG}"]

df["is_black"] = (~np.isfinite(df["payback_years"])) | (df["payback_years"] > PAYBACK_HORIZON)

df["payback_clip"] = pd.to_numeric(df["payback_years"], errors="coerce")
df.loc[~np.isfinite(df["payback_clip"]), "payback_clip"] = PAYBACK_HORIZON
df["payback_clip"] = df["payback_clip"].clip(lower=0, upper=PAYBACK_HORIZON)

type_cols = [c for c in ["park_class", "park_class_name"] if c in df.columns]

type_summary_frames = []
if type_cols:
    for r in DISCOUNT_RATES:
        rate_tag = rate_tag_from_r(r)
        rate_pct = int(round(r * 100))

        summary_one = (
            df.groupby(type_cols, dropna=False, as_index=False)
            .agg(
                parks_n=("osm_id_norm", "count"),
                annual_benefit_sum_2019=("annual_benefit_yen_2019", "sum"),
                annual_om_sum_2019=("annual_om_cost_2019", "sum"),
                annual_net_sum_2019=("annual_net_benefit_2019", "sum"),
                investment_sum_2019=("one_time_investment_2019", "sum"),
                npv_50y_sum=(f"npv_50y_{rate_tag}", "sum"),
                recoup_share_50y=(f"recoup_within_50y_{rate_tag}", "mean"),
                never_payback_share=(f"discounted_payback_years_{rate_tag}", lambda s: np.isinf(pd.to_numeric(s, errors='coerce')).mean()),

                median_full_payback_years_finite_only=(f"discounted_payback_years_{rate_tag}", finite_median),
                mean_full_payback_years_finite_only=(f"discounted_payback_years_{rate_tag}", finite_mean),

                median_full_payback_years_capped500=(f"discounted_payback_years_{rate_tag}", lambda s: capped_median(s, cap=FULL_PAYBACK_CAP_FOR_SUMMARY)),
                mean_full_payback_years_capped500=(f"discounted_payback_years_{rate_tag}", lambda s: capped_mean(s, cap=FULL_PAYBACK_CAP_FOR_SUMMARY)),

                median_payback_within_50y=(f"discounted_payback_years_50y_{rate_tag}", "median"),
                mean_payback_within_50y=(f"discounted_payback_years_50y_{rate_tag}", "mean"),
            )
        )
        summary_one["discount_rate"] = r
        summary_one["discount_rate_pct"] = rate_pct
        type_summary_frames.append(summary_one)

if type_summary_frames:
    type_summary = pd.concat(type_summary_frames, ignore_index=True)
else:
    type_summary = pd.DataFrame()

benefit_backcheck = np.nan
mask_b = df["steps_sum"] > 0
if mask_b.any():
    benefit_backcheck = np.nanmedian(
        df.loc[mask_b, "annual_benefit_yen_2019"]
        / (df.loc[mask_b, "steps_sum"] * EXPANSION_FACTOR)
    )

match_rate_land = 1 - df["land_cost_2019"].isna().mean()
match_rate_om = 1 - df["om_unit_cost_2019"].isna().mean()
match_rate_area = 1 - df["area_m2"].isna().mean()

diag_lines = []
diag_lines.append(f"visit rows: {len(visits):,}")
diag_lines.append(f"park rows after aggregation: {len(parks):,}")
diag_lines.append(f"final park rows: {len(df):,}")
diag_lines.append("")
diag_lines.append(f"evaluation years: {EVAL_YEARS}")
diag_lines.append(f"discount rates: {DISCOUNT_RATES}")
diag_lines.append(f"main discount rate: {MAIN_RATE:.2%}")
diag_lines.append(f"yen per step (2019 JPY): {YEN_PER_STEP_2019}")
diag_lines.append(f"expansion factor: {EXPANSION_FACTOR}")
diag_lines.append("")
diag_lines.append(f"construction unit cost (2014): {CONSTRUCTION_UNIT_COST_2014:,.2f} JPY/m2")
diag_lines.append(f"construction to 2019 factor: {CONSTRUCTION_TO_2019_FACTOR:.6f}")
diag_lines.append(f"O&M to 2019 factor: {OM_TO_2019_FACTOR:.6f}")
diag_lines.append("")
diag_lines.append(f"match rate land_cost_2019: {match_rate_land:.2%}")
diag_lines.append(f"match rate om_unit_cost_2019: {match_rate_om:.2%}")
diag_lines.append(f"match rate area_m2: {match_rate_area:.2%}")
diag_lines.append("")
diag_lines.append(f"benefit back-check (should be ~{YEN_PER_STEP_2019}): {benefit_backcheck}")
diag_lines.append(f"simple payback total <=50y share: {(df['simple_payback_years_total'] <= EVAL_YEARS).mean():.2%}")
diag_lines.append(f"simple payback land only <=50y share: {(df['simple_payback_years_land_only'] <= EVAL_YEARS).mean():.2%}")
diag_lines.append("")

for r in DISCOUNT_RATES:
    rate_tag = rate_tag_from_r(r)
    rate_pct = int(round(r * 100))

    diag_lines.append(f"----- rate = {rate_pct}% -----")
    diag_lines.append(
        f"recoup within 50y share @ {rate_pct}%: "
        f"{df[f'recoup_within_50y_{rate_tag}'].mean():.2%}"
    )
    diag_lines.append(
        f"NPV>0 share @ {rate_pct}%: "
        f"{(df[f'npv_50y_{rate_tag}'] > 0).mean():.2%}"
    )
    diag_lines.append(
        f"never payback share @ {rate_pct}%: "
        f"{np.isinf(pd.to_numeric(df[f'discounted_payback_years_{rate_tag}'], errors='coerce')).mean():.2%}"
    )

ensure_dir(OUT_DIR)

front_cols = [
    "osm_id_norm",
    "park_class",
    "park_class_name",
    "area_m2",
    "steps_sum",
    "annual_benefit_yen_2019",
    "om_unit_cost_raw",
    "om_unit_cost_2019",
    "annual_om_cost_2019",
    "annual_net_benefit_2019",
    "land_cost_2019",
    "construction_cost_2019",
    "one_time_investment_2019",
    "simple_payback_years_total",
    "simple_payback_years_land_only",

    "discounted_payback_years_r0",
    "discounted_payback_years_r1",
    "discounted_payback_years_r2",
    "discounted_payback_years_r4",

    "discounted_payback_years_50y_r0",
    "discounted_payback_years_50y_r1",
    "discounted_payback_years_50y_r2",
    "discounted_payback_years_50y_r4",

    "npv_50y_r0",
    "npv_50y_r1",
    "npv_50y_r2",
    "npv_50y_r4",

    "payback_years",
    "payback_years_50y",
    "discounted_payback_status",
    "npv_50y_main",
    "is_black",
    "payback_clip",
    "Lng",
    "Lat",
]
keep_cols = [c for c in front_cols if c in df.columns] + [c for c in df.columns if c not in front_cols]
df = df[keep_cols].copy()

df.to_csv(OUT_TABLE_CSV, index=False, encoding="utf-8-sig")
print("Saved:", OUT_TABLE_CSV)

if not type_summary.empty:
    type_summary.to_csv(OUT_TYPE_SUMMARY_CSV, index=False, encoding="utf-8-sig")
    print("Saved:", OUT_TYPE_SUMMARY_CSV)

rate_summary.to_csv(OUT_RATE_SUMMARY_CSV, index=False, encoding="utf-8-sig")
print("Saved:", OUT_RATE_SUMMARY_CSV)

with open(OUT_DIAG_TXT, "w", encoding="utf-8") as f:
    f.write("\n".join(diag_lines))
print("Saved:", OUT_DIAG_TXT)

print("\n".join(diag_lines))
print(f"Cell1 done in {time.perf_counter()-t0:.1f}s")

from IPython.display import display

print("\nRate summary:")
display(rate_summary)

print("\nPreview of df:")
display(df.head())


In [ ]:
# Calculate the alternative 2% main discount-rate results.
import os
import time
import numpy as np
import pandas as pd
from IPython.display import display

t0 = time.perf_counter()

VISIT_FILES = [
    r"data/restricted/mobility/mobility_part_0.csv",
    r"data/restricted/mobility/mobility_part_1.csv",
]

COST_XLSX = (
    r"data/restricted/park_costs/park_costs_2019_2024.xlsx"
)

OUT_DIR = (
    r"outputs"
)

OUT_TABLE_CSV = os.path.join(OUT_DIR, "park_payback_discounted_multi_rate_fullpayback_2019_50y_main2pct.csv")
OUT_TYPE_SUMMARY_CSV = os.path.join(OUT_DIR, "park_payback_discounted_multi_rate_fullpayback_2019_50y_by_type_main2pct.csv")
OUT_RATE_SUMMARY_CSV = os.path.join(OUT_DIR, "park_discount_rate_sensitivity_fullpayback_summary_main2pct.csv")
OUT_DIAG_TXT = os.path.join(OUT_DIR, "park_payback_discounted_multi_rate_fullpayback_2019_50y_diagnostics_main2pct.txt")

EVAL_YEARS = 50
DISCOUNT_RATES = [0.01, 0.02, 0.04]
MAIN_RATE = 0.02
PAYBACK_HORIZON = 50

YEN_PER_STEP_2019 = 0.04056
EXPANSION_FACTOR = 150

CONSTRUCTION_UNIT_COST_2014 = 12000.0
AREA_IS_HECTARE_WHEN_ONLY_AREA = False

FULL_PAYBACK_CAP_FOR_SUMMARY = 500.0

CPI_2003_2015BASE = 97.2
CPI_2014_2015BASE = 102.8 / 103.6 * 100.0
CPI_2019_2015BASE = 101.8

CONSTRUCTION_TO_2019_FACTOR = CPI_2019_2015BASE / CPI_2014_2015BASE
OM_TO_2019_FACTOR = CPI_2019_2015BASE / CPI_2003_2015BASE

def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def norm_osm(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\.0$", "", regex=True)
    s = s.replace({"nan": np.nan, "None": np.nan, "": np.nan})
    return s

def first_valid(series: pd.Series):
    s = series.dropna()
    return s.iloc[0] if len(s) else np.nan

def numeric_median(series: pd.Series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    return float(s.median()) if len(s) else np.nan

def pick_column(columns, exact_candidates=(), contains_all=(), contains_any=(),
                required=True, label="column"):
    cols = list(columns)
    lower_map = {c.lower(): c for c in cols}

    for cand in exact_candidates:
        if cand in cols:
            return cand
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]

    for c in cols:
        cl = c.lower()
        ok_all = all(k.lower() in cl for k in contains_all) if contains_all else True
        ok_any = any(k.lower() in cl for k in contains_any) if contains_any else True
        if ok_all and ok_any:
            return c

    if required:
        raise KeyError(f"Cannot find {label}. Available columns:\n{cols}")
    return None

def rate_tag_from_r(r: float) -> str:
    return f"r{int(round(r * 100))}"

def annuity_factor(r: float, years: int) -> float:
    if r == 0:
        return float(years)
    return float(np.sum(1.0 / (1.0 + r) ** np.arange(1, years + 1)))

def full_payback_year_array(invest_arr, annual_net_arr, r):
    invest = np.asarray(invest_arr, dtype=float)
    annual_net = np.asarray(annual_net_arr, dtype=float)

    out = np.full(len(invest), np.nan, dtype=float)
    valid_base = np.isfinite(invest) & np.isfinite(annual_net)

    mask_zero_inv = valid_base & (invest <= 0)
    out[mask_zero_inv] = 0.0

    mask_never_nonpos_net = valid_base & (invest > 0) & (annual_net <= 0)
    out[mask_never_nonpos_net] = np.inf

    mask_work = valid_base & (invest > 0) & (annual_net > 0)
    if not np.any(mask_work):
        return out

    I = invest[mask_work]
    N = annual_net[mask_work]

    if r == 0:
        out[mask_work] = np.ceil(I / N)
        return out

    mask_finite = I < (N / r)
    idx_work = np.where(mask_work)[0]

    idx_never = idx_work[~mask_finite]
    out[idx_never] = np.inf

    if np.any(mask_finite):
        I_fin = I[mask_finite]
        N_fin = N[mask_finite]
        a = r * I_fin / N_fin
        T = np.ceil(-np.log1p(-a) / np.log1p(r))
        idx_fin = idx_work[mask_finite]
        out[idx_fin] = T

    return out

def payback_within_horizon_array(full_payback_arr, years):
    arr = np.asarray(full_payback_arr, dtype=float)
    return np.where(np.isfinite(arr) & (arr <= years), arr, np.nan)

def capped_payback_stats(arr, cap=500.0):
    x = pd.to_numeric(pd.Series(arr), errors="coerce").to_numpy(dtype=float)
    x = np.where(np.isinf(x), cap, x)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan
    return float(np.median(x)), float(np.mean(x))

def finite_mean(s):
    x = pd.to_numeric(s, errors="coerce").to_numpy(dtype=float)
    x = x[np.isfinite(x)]
    return float(np.mean(x)) if len(x) else np.nan

def finite_median(s):
    x = pd.to_numeric(s, errors="coerce").to_numpy(dtype=float)
    x = x[np.isfinite(x)]
    return float(np.median(x)) if len(x) else np.nan

def capped_mean(s, cap=500.0):
    x = pd.to_numeric(s, errors="coerce").to_numpy(dtype=float)
    x = np.where(np.isinf(x), cap, x)
    x = x[np.isfinite(x)]
    return float(np.mean(x)) if len(x) else np.nan

def capped_median(s, cap=500.0):
    x = pd.to_numeric(s, errors="coerce").to_numpy(dtype=float)
    x = np.where(np.isinf(x), cap, x)
    x = x[np.isfinite(x)]
    return float(np.median(x)) if len(x) else np.nan

frames = []
for fp in VISIT_FILES:
    if not os.path.exists(fp):
        raise FileNotFoundError(fp)
    tmp = pd.read_csv(fp, low_memory=False)
    tmp["__source_file__"] = os.path.basename(fp)
    frames.append(tmp)

visits = pd.concat(frames, ignore_index=True)
print("visit rows:", f"{len(visits):,}")

osm_col_vis = pick_column(
    visits.columns,
    exact_candidates=["osm_id"],
    contains_all=["osm", "id"],
    label="visit osm_id",
)
steps_col = pick_column(
    visits.columns,
    exact_candidates=["steps"],
    contains_any=["steps", "step"],
    label="visit steps",
)

area_m2_col_vis = None
if "area_m2" in visits.columns:
    area_m2_col_vis = "area_m2"
elif "area" in visits.columns:
    area_m2_col_vis = "area"

park_class_col = "park_class" if "park_class" in visits.columns else None
park_class_name_col = "park_class_name" if "park_class_name" in visits.columns else None

lon_col_vis = None
lat_col_vis = None
lon_candidates = [c for c in ["Lng", "lng", "lon", "Lon", "longitude", "Longitude",
                              "park_lng", "park_lon", "target_lng", "centroid_lng"] if c in visits.columns]
lat_candidates = [c for c in ["Lat", "lat", "latitude", "Latitude",
                              "park_lat", "target_lat", "centroid_lat"] if c in visits.columns]
if lon_candidates and lat_candidates:
    lon_col_vis = lon_candidates[0]
    lat_col_vis = lat_candidates[0]

visits["osm_id_norm"] = norm_osm(visits[osm_col_vis])
visits["steps_num"] = pd.to_numeric(visits[steps_col], errors="coerce").fillna(0.0)

if area_m2_col_vis is not None:
    visits["area_m2_tmp"] = pd.to_numeric(visits[area_m2_col_vis], errors="coerce")
    if area_m2_col_vis == "area" and AREA_IS_HECTARE_WHEN_ONLY_AREA:
        visits["area_m2_tmp"] = visits["area_m2_tmp"] * 10000.0
else:
    visits["area_m2_tmp"] = np.nan

agg_dict = {
    "steps_num": "sum",
    "area_m2_tmp": numeric_median,
}
rename_dict = {
    "steps_num": "steps_sum",
    "area_m2_tmp": "area_m2",
}

if park_class_col:
    agg_dict[park_class_col] = first_valid
    rename_dict[park_class_col] = "park_class"
if park_class_name_col:
    agg_dict[park_class_name_col] = first_valid
    rename_dict[park_class_name_col] = "park_class_name"
if lon_col_vis and lat_col_vis:
    agg_dict[lon_col_vis] = numeric_median
    agg_dict[lat_col_vis] = numeric_median
    rename_dict[lon_col_vis] = "Lng"
    rename_dict[lat_col_vis] = "Lat"

parks = (
    visits.dropna(subset=["osm_id_norm"])
    .groupby("osm_id_norm", as_index=False)
    .agg(agg_dict)
    .rename(columns=rename_dict)
)
print("parks after visit aggregation:", f"{len(parks):,}")

if not os.path.exists(COST_XLSX):
    raise FileNotFoundError(COST_XLSX)

cost = pd.read_excel(COST_XLSX, engine="openpyxl")
print("cost rows:", f"{len(cost):,}")

osm_col_cost = pick_column(
    cost.columns,
    exact_candidates=["osm_id"],
    contains_all=["osm", "id"],
    label="cost osm_id",
)

land_total_col = pick_column(
    cost.columns,
    exact_candidates=["land_price_1year"],
    contains_all=["land", "price", "1year"],
    required=False,
    label="land total cost column",
)
if land_total_col is None:
    raise KeyError("land_price_1year was not found.")

om_unit_col = pick_column(
    cost.columns,
    exact_candidates=[
        "unit_maintenance_yen_per_m2_2024",
        "unit_maintenance_yen_per_m2",
        "annual_om_unit_yen_per_m2",
        "O&M单价",
        "养护单价",
        "维护单价",
        "管理单价",
    ],
    contains_any=["maintenance", "om", "o&m", "养护", "维护", "管理"],
    label="O&M unit cost column",
)

area_m2_col_cost = None
if "area_m2" in cost.columns:
    area_m2_col_cost = "area_m2"
elif "area" in cost.columns:
    area_m2_col_cost = "area"

cost["osm_id_norm"] = norm_osm(cost[osm_col_cost])
cost["land_price_1year_raw"] = pd.to_numeric(cost[land_total_col], errors="coerce")
cost["om_unit_cost_raw"] = pd.to_numeric(cost[om_unit_col], errors="coerce")

if area_m2_col_cost is not None:
    cost["area_m2_cost"] = pd.to_numeric(cost[area_m2_col_cost], errors="coerce")
    if area_m2_col_cost == "area" and AREA_IS_HECTARE_WHEN_ONLY_AREA:
        cost["area_m2_cost"] = cost["area_m2_cost"] * 10000.0
else:
    cost["area_m2_cost"] = np.nan

cost["land_cost_2019"] = cost["land_price_1year_raw"]
cost["om_unit_cost_2019"] = cost["om_unit_cost_raw"] * OM_TO_2019_FACTOR
cost = cost.drop_duplicates("osm_id_norm", keep="last")

df = parks.merge(
    cost[
        [
            "osm_id_norm",
            "land_cost_2019",
            "om_unit_cost_raw",
            "om_unit_cost_2019",
            "area_m2_cost",
        ]
    ],
    on="osm_id_norm",
    how="left",
)

df["area_m2"] = pd.to_numeric(df["area_m2"], errors="coerce")
df["area_m2"] = df["area_m2"].fillna(df["area_m2_cost"])

df["steps_sum"] = pd.to_numeric(df["steps_sum"], errors="coerce").fillna(0.0)

df["annual_benefit_yen_2019"] = df["steps_sum"] * YEN_PER_STEP_2019 * EXPANSION_FACTOR
df["construction_cost_2019"] = df["area_m2"] * CONSTRUCTION_UNIT_COST_2014 * CONSTRUCTION_TO_2019_FACTOR
df["annual_om_cost_2019"] = df["om_unit_cost_2019"] * df["area_m2"]
df["annual_net_benefit_2019"] = df["annual_benefit_yen_2019"] - df["annual_om_cost_2019"]
df["one_time_investment_2019"] = df["land_cost_2019"] + df["construction_cost_2019"]

df["simple_payback_years_total"] = np.where(
    (df["annual_net_benefit_2019"] > 0)
    & np.isfinite(df["annual_net_benefit_2019"])
    & np.isfinite(df["one_time_investment_2019"]),
    df["one_time_investment_2019"] / df["annual_net_benefit_2019"],
    np.nan,
)

df["simple_payback_years_land_only"] = np.where(
    (df["annual_net_benefit_2019"] > 0)
    & np.isfinite(df["annual_net_benefit_2019"])
    & np.isfinite(df["land_cost_2019"]),
    df["land_cost_2019"] / df["annual_net_benefit_2019"],
    np.nan,
)

invest_arr = pd.to_numeric(df["one_time_investment_2019"], errors="coerce").to_numpy(dtype=float)
annual_net_arr = pd.to_numeric(df["annual_net_benefit_2019"], errors="coerce").to_numpy(dtype=float)

rate_summary_rows = []

for r in DISCOUNT_RATES:
    rate_tag = rate_tag_from_r(r)
    rate_pct = int(round(r * 100))

    pv_factor = annuity_factor(r, EVAL_YEARS)
    df[f"pv_factor_50y_{rate_tag}"] = pv_factor

    df[f"pv_annual_net_benefit_50y_{rate_tag}"] = df["annual_net_benefit_2019"] * pv_factor
    df[f"npv_50y_{rate_tag}"] = -df["one_time_investment_2019"] + df[f"pv_annual_net_benefit_50y_{rate_tag}"]

    full_payback_arr = full_payback_year_array(
        invest_arr=invest_arr,
        annual_net_arr=annual_net_arr,
        r=r
    )
    df[f"discounted_payback_years_{rate_tag}"] = full_payback_arr

    within_50_arr = payback_within_horizon_array(full_payback_arr, EVAL_YEARS)
    df[f"discounted_payback_years_50y_{rate_tag}"] = within_50_arr
    df[f"recoup_within_50y_{rate_tag}"] = np.isfinite(within_50_arr)

    full_status = np.full(len(df), "invalid", dtype=object)
    full_status[np.isfinite(full_payback_arr)] = "finite payback"
    full_status[np.isinf(full_payback_arr)] = "never pay back"
    df[f"discounted_payback_status_full_{rate_tag}"] = full_status

    status_50y = np.full(len(df), f"not recouped within {EVAL_YEARS} years", dtype=object)
    status_50y[np.isfinite(within_50_arr)] = f"recouped within {EVAL_YEARS} years"
    status_50y[np.isinf(full_payback_arr)] = "never pay back"
    df[f"discounted_payback_status_50y_{rate_tag}"] = status_50y

    finite_full = pd.Series(full_payback_arr)[np.isfinite(full_payback_arr)]
    median_capped, mean_capped = capped_payback_stats(full_payback_arr, cap=FULL_PAYBACK_CAP_FOR_SUMMARY)

    rate_summary_rows.append({
        "discount_rate": r,
        "discount_rate_pct": rate_pct,
        "pv_factor_50y": pv_factor,
        "recoup_share_50y": float(np.isfinite(within_50_arr).mean()),
        "npv_positive_share": float((df[f"npv_50y_{rate_tag}"] > 0).mean()),
        "never_payback_share": float(np.isinf(full_payback_arr).mean()),
        "median_full_payback_years_finite_only": float(finite_full.median()) if len(finite_full) else np.nan,
        "mean_full_payback_years_finite_only": float(finite_full.mean()) if len(finite_full) else np.nan,
        f"median_full_payback_years_capped{int(FULL_PAYBACK_CAP_FOR_SUMMARY)}": median_capped,
        f"mean_full_payback_years_capped{int(FULL_PAYBACK_CAP_FOR_SUMMARY)}": mean_capped,
        "median_payback_within_50y": float(pd.Series(within_50_arr).median()) if np.isfinite(within_50_arr).any() else np.nan,
        "mean_payback_within_50y": float(pd.Series(within_50_arr).mean()) if np.isfinite(within_50_arr).any() else np.nan,
    })

rate_summary = pd.DataFrame(rate_summary_rows)

MAIN_RATE_TAG = rate_tag_from_r(MAIN_RATE)

df["payback_years"] = df[f"discounted_payback_years_{MAIN_RATE_TAG}"]
df["payback_years_50y"] = df[f"discounted_payback_years_50y_{MAIN_RATE_TAG}"]
df["discounted_payback_status"] = df[f"discounted_payback_status_50y_{MAIN_RATE_TAG}"]
df["npv_50y_main"] = df[f"npv_50y_{MAIN_RATE_TAG}"]

df["is_black"] = (~np.isfinite(df["payback_years"])) | (df["payback_years"] > PAYBACK_HORIZON)

df["payback_clip"] = pd.to_numeric(df["payback_years"], errors="coerce")
df.loc[~np.isfinite(df["payback_clip"]), "payback_clip"] = PAYBACK_HORIZON
df["payback_clip"] = df["payback_clip"].clip(lower=0, upper=PAYBACK_HORIZON)

type_cols = [c for c in ["park_class", "park_class_name"] if c in df.columns]

type_summary_frames = []
if type_cols:
    for r in DISCOUNT_RATES:
        rate_tag = rate_tag_from_r(r)
        rate_pct = int(round(r * 100))

        summary_one = (
            df.groupby(type_cols, dropna=False, as_index=False)
            .agg(
                parks_n=("osm_id_norm", "count"),
                annual_benefit_sum_2019=("annual_benefit_yen_2019", "sum"),
                annual_om_sum_2019=("annual_om_cost_2019", "sum"),
                annual_net_sum_2019=("annual_net_benefit_2019", "sum"),
                investment_sum_2019=("one_time_investment_2019", "sum"),
                npv_50y_sum=(f"npv_50y_{rate_tag}", "sum"),
                recoup_share_50y=(f"recoup_within_50y_{rate_tag}", "mean"),
                never_payback_share=(f"discounted_payback_years_{rate_tag}", lambda s: np.isinf(pd.to_numeric(s, errors='coerce')).mean()),
                median_full_payback_years_finite_only=(f"discounted_payback_years_{rate_tag}", finite_median),
                mean_full_payback_years_finite_only=(f"discounted_payback_years_{rate_tag}", finite_mean),
                median_full_payback_years_capped500=(f"discounted_payback_years_{rate_tag}", lambda s: capped_median(s, cap=FULL_PAYBACK_CAP_FOR_SUMMARY)),
                mean_full_payback_years_capped500=(f"discounted_payback_years_{rate_tag}", lambda s: capped_mean(s, cap=FULL_PAYBACK_CAP_FOR_SUMMARY)),
                median_payback_within_50y=(f"discounted_payback_years_50y_{rate_tag}", "median"),
                mean_payback_within_50y=(f"discounted_payback_years_50y_{rate_tag}", "mean"),
            )
        )
        summary_one["discount_rate"] = r
        summary_one["discount_rate_pct"] = rate_pct
        type_summary_frames.append(summary_one)

type_summary = pd.concat(type_summary_frames, ignore_index=True) if type_summary_frames else pd.DataFrame()

benefit_backcheck = np.nan
mask_b = df["steps_sum"] > 0
if mask_b.any():
    benefit_backcheck = np.nanmedian(
        df.loc[mask_b, "annual_benefit_yen_2019"] /
        (df.loc[mask_b, "steps_sum"] * EXPANSION_FACTOR)
    )

match_rate_land = 1 - df["land_cost_2019"].isna().mean()
match_rate_om = 1 - df["om_unit_cost_2019"].isna().mean()
match_rate_area = 1 - df["area_m2"].isna().mean()

diag_lines = []
diag_lines.append(f"visit rows: {len(visits):,}")
diag_lines.append(f"park rows after visit aggregation: {len(parks):,}")
diag_lines.append(f"final park rows: {len(df):,}")
diag_lines.append("")
diag_lines.append(f"evaluation years: {EVAL_YEARS}")
diag_lines.append(f"discount rates: {DISCOUNT_RATES}")
diag_lines.append(f"main discount rate: {MAIN_RATE:.2%}")
diag_lines.append(f"yen per step (2019 JPY): {YEN_PER_STEP_2019}")
diag_lines.append(f"expansion factor: {EXPANSION_FACTOR}")
diag_lines.append("")
diag_lines.append(f"construction unit cost (2014): {CONSTRUCTION_UNIT_COST_2014:,.2f} JPY/m2")
diag_lines.append(f"construction to 2019 factor: {CONSTRUCTION_TO_2019_FACTOR:.6f}")
diag_lines.append(f"O&M to 2019 factor: {OM_TO_2019_FACTOR:.6f}")
diag_lines.append("")
diag_lines.append(f"match rate land_cost_2019: {match_rate_land:.2%}")
diag_lines.append(f"match rate om_unit_cost_2019: {match_rate_om:.2%}")
diag_lines.append(f"match rate area_m2: {match_rate_area:.2%}")
diag_lines.append("")
diag_lines.append(f"benefit back-check (should be ~{YEN_PER_STEP_2019}): {benefit_backcheck}")
diag_lines.append(f"simple payback total <=50y share: {(df['simple_payback_years_total'] <= EVAL_YEARS).mean():.2%}")
diag_lines.append(f"simple payback land only <=50y share: {(df['simple_payback_years_land_only'] <= EVAL_YEARS).mean():.2%}")
diag_lines.append("")

for r in DISCOUNT_RATES:
    rate_tag = rate_tag_from_r(r)
    rate_pct = int(round(r * 100))
    diag_lines.append(f"----- rate = {rate_pct}% -----")
    diag_lines.append(f"recoup within 50y share @ {rate_pct}%: {df[f'recoup_within_50y_{rate_tag}'].mean():.2%}")
    diag_lines.append(f"NPV>0 share @ {rate_pct}%: {(df[f'npv_50y_{rate_tag}'] > 0).mean():.2%}")
    diag_lines.append(f"never payback share @ {rate_pct}%: {np.isinf(pd.to_numeric(df[f'discounted_payback_years_{rate_tag}'], errors='coerce')).mean():.2%}")

ensure_dir(OUT_DIR)

front_cols = [
    "osm_id_norm",
    "park_class",
    "park_class_name",
    "area_m2",
    "steps_sum",
    "annual_benefit_yen_2019",
    "om_unit_cost_raw",
    "om_unit_cost_2019",
    "annual_om_cost_2019",
    "annual_net_benefit_2019",
    "land_cost_2019",
    "construction_cost_2019",
    "one_time_investment_2019",
    "simple_payback_years_total",
    "simple_payback_years_land_only",
    "discounted_payback_years_r1",
    "discounted_payback_years_r2",
    "discounted_payback_years_r4",
    "discounted_payback_years_50y_r1",
    "discounted_payback_years_50y_r2",
    "discounted_payback_years_50y_r4",
    "npv_50y_r1",
    "npv_50y_r2",
    "npv_50y_r4",
    "payback_years",
    "payback_years_50y",
    "discounted_payback_status",
    "npv_50y_main",
    "is_black",
    "payback_clip",
    "Lng",
    "Lat",
]
keep_cols = [c for c in front_cols if c in df.columns] + [c for c in df.columns if c not in front_cols]
df = df[keep_cols].copy()

df.to_csv(OUT_TABLE_CSV, index=False, encoding="utf-8-sig")
print("Saved:", OUT_TABLE_CSV)

if not type_summary.empty:
    type_summary.to_csv(OUT_TYPE_SUMMARY_CSV, index=False, encoding="utf-8-sig")
    print("Saved:", OUT_TYPE_SUMMARY_CSV)

rate_summary.to_csv(OUT_RATE_SUMMARY_CSV, index=False, encoding="utf-8-sig")
print("Saved:", OUT_RATE_SUMMARY_CSV)

with open(OUT_DIAG_TXT, "w", encoding="utf-8") as f:
    f.write("\n".join(diag_lines))
print("Saved:", OUT_DIAG_TXT)

print("\n".join(diag_lines))
print(f"\nDone in {time.perf_counter() - t0:.1f}s")

print("\nRate summary:")
display(rate_summary)

print("\nPreview of output table:")
display(df.head())


In [ ]:
# Calculate discounted payback with benefit and maintenance-cost growth.
import os
import time
import numpy as np
import pandas as pd

t0 = time.perf_counter()

VISIT_FILES = [
    r"data/restricted/mobility/mobility_part_0.csv",
    r"data/restricted/mobility/mobility_part_1.csv",
]

COST_XLSX = (
    r"data/restricted/park_costs/park_costs_2019_2024.xlsx"
)

OUT_DIR = (
    r"outputs"
)
os.makedirs(OUT_DIR, exist_ok=True)

OUT_TABLE_CSV = os.path.join(
    OUT_DIR,
    "park_payback_growth_discounted_2019_50y_r2_b1_om1.csv"
)
OUT_TYPE_SUMMARY_CSV = os.path.join(
    OUT_DIR,
    "park_payback_growth_discounted_2019_50y_by_type_r2_b1_om1.csv"
)
OUT_DIAG_TXT = os.path.join(
    OUT_DIR,
    "park_payback_growth_discounted_2019_50y_diagnostics_r2_b1_om1.txt"
)

BASE_YEAR = 2019
EVAL_YEARS = 50
DISCOUNT_RATE = 0.02

BENEFIT_GROWTH = 0.02
OM_GROWTH = 0.02

YEN_PER_STEP_2019 = 0.04056
EXPANSION_FACTOR = 150

CONSTRUCTION_UNIT_COST_2014 = 12000.0
AREA_IS_HECTARE_WHEN_ONLY_AREA = False

CPI_2003_2015BASE = 97.2
CPI_2014_2015BASE = 102.8 / 103.6 * 100.0
CPI_2019_2015BASE = 101.8

CONSTRUCTION_TO_2019_FACTOR = CPI_2019_2015BASE / CPI_2014_2015BASE
OM_TO_2019_FACTOR = CPI_2019_2015BASE / CPI_2003_2015BASE

def norm_osm(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\.0$", "", regex=True)
    s = s.replace({"nan": np.nan, "None": np.nan, "": np.nan})
    return s

def first_valid(series: pd.Series):
    s = series.dropna()
    return s.iloc[0] if len(s) else np.nan

def numeric_median(series: pd.Series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    return float(s.median()) if len(s) else np.nan

def pick_column(columns, exact_candidates=(), contains_all=(), contains_any=(),
                required=True, label="column"):
    cols = list(columns)
    lower_map = {c.lower(): c for c in cols}

    for cand in exact_candidates:
        if cand in cols:
            return cand
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]

    for c in cols:
        cl = c.lower()
        ok_all = all(k.lower() in cl for k in contains_all) if contains_all else True
        ok_any = any(k.lower() in cl for k in contains_any) if contains_any else True
        if ok_all and ok_any:
            return c

    if required:
        raise KeyError(f"Cannot find {label}. Available columns:\n{cols}")
    return None

def classify_payback_year(y):
    if pd.isna(y):
        return "invalid"
    if np.isinf(y):
        return "never"
    if y <= 5:
        return "0-5"
    elif y <= 20:
        return "5-20"
    elif y <= 50:
        return "20-50"
    else:
        return ">50"

def discounted_payback_with_growth(invest, annual_benefit_0, annual_om_0,
                                   r, g_b, g_om, years=50):
    """
    Simulate annual discounted cash flow with independently growing benefits
    and maintenance costs. The initial investment occurs at year zero; full
    payback may extend beyond 50 years, while the horizon-specific result is
    reported only when recovery occurs within 50 years.
    """
    if not np.isfinite(invest) or not np.isfinite(annual_benefit_0) or not np.isfinite(annual_om_0):
        return np.nan, np.nan, np.nan, "invalid"

    if invest <= 0:
        npv_50y = 0.0
        # Benefits and maintenance costs grow independently before annual discounting.
        for t in range(1, years + 1):
            benefit_t = annual_benefit_0 * ((1 + g_b) ** (t - 1))
            om_t = annual_om_0 * ((1 + g_om) ** (t - 1))
            net_t = benefit_t - om_t
            npv_50y += net_t / ((1 + r) ** t)
        return 0.0, 0.0, npv_50y, "recouped within 50 years"

    cum_pv = 0.0
    payback_year = np.nan
    npv_50y = -invest

    for t in range(1, years + 1):
        benefit_t = annual_benefit_0 * ((1 + g_b) ** (t - 1))
        om_t = annual_om_0 * ((1 + g_om) ** (t - 1))
        net_t = benefit_t - om_t
        pv_t = net_t / ((1 + r) ** t)

        cum_pv += pv_t
        npv_50y += pv_t

        if pd.isna(payback_year) and cum_pv >= invest:
            payback_year = float(t)

    if pd.notna(payback_year):
        return payback_year, payback_year, npv_50y, "recouped within 50 years"

    cum_pv_long = cum_pv
    max_year_long = 500
    payback_year_long = np.nan

    for t in range(years + 1, max_year_long + 1):
        benefit_t = annual_benefit_0 * ((1 + g_b) ** (t - 1))
        om_t = annual_om_0 * ((1 + g_om) ** (t - 1))
        net_t = benefit_t - om_t
        pv_t = net_t / ((1 + r) ** t)

        cum_pv_long += pv_t

        if pd.isna(payback_year_long) and cum_pv_long >= invest:
            payback_year_long = float(t)
            break

    if pd.notna(payback_year_long):
        return payback_year_long, np.nan, npv_50y, "not recouped within 50 years"
    else:
        return np.inf, np.nan, npv_50y, "never pay back"

visit_frames = []
for fp in VISIT_FILES:
    df0 = pd.read_csv(fp, low_memory=False)
    visit_frames.append(df0)

visits = pd.concat(visit_frames, ignore_index=True)

osm_col = pick_column(
    visits.columns,
    exact_candidates=("osm_id",),
    contains_any=("osm_id",),
    label="osm_id"
)
steps_col = pick_column(
    visits.columns,
    exact_candidates=("steps",),
    contains_any=("steps",),
    label="steps"
)
lat_col = pick_column(
    visits.columns,
    exact_candidates=("Lat", "lat", "latitude"),
    contains_any=("lat",),
    label="lat"
)
lng_col = pick_column(
    visits.columns,
    exact_candidates=("Lng", "lng", "lon", "longitude"),
    contains_any=("lng", "lon"),
    label="lng"
)
area_col = pick_column(
    visits.columns,
    exact_candidates=("area_m2", "area"),
    contains_any=("area",),
    label="area"
)
park_class_col = pick_column(
    visits.columns,
    exact_candidates=("park_class",),
    contains_all=("park", "class"),
    contains_any=("class",),
    required=False,
    label="park_class"
)
park_class_name_col = pick_column(
    visits.columns,
    exact_candidates=("park_class_name",),
    contains_all=("park", "class", "name"),
    contains_any=("class_name", "park_class_name"),
    required=False,
    label="park_class_name"
)

visits = visits.copy()
visits["osm_id_norm"] = norm_osm(visits[osm_col])
visits["steps_num"] = pd.to_numeric(visits[steps_col], errors="coerce")
visits["lat_num"] = pd.to_numeric(visits[lat_col], errors="coerce")
visits["lng_num"] = pd.to_numeric(visits[lng_col], errors="coerce")
visits["area_raw"] = pd.to_numeric(visits[area_col], errors="coerce")

if AREA_IS_HECTARE_WHEN_ONLY_AREA:
    visits["area_m2"] = visits["area_raw"] * 10000.0
else:
    visits["area_m2"] = visits["area_raw"]

group_cols = ["osm_id_norm"]

agg_dict = {
    "lat_num": "mean",
    "lng_num": "mean",
    "area_m2": numeric_median,
    "steps_num": "sum",
}

if park_class_col is not None:
    agg_dict[park_class_col] = first_valid
if park_class_name_col is not None:
    agg_dict[park_class_name_col] = first_valid

park_agg = (
    visits.loc[visits["osm_id_norm"].notna()]
    .groupby(group_cols, dropna=False, as_index=False)
    .agg(agg_dict)
)

park_agg = park_agg.rename(columns={
    "lat_num": "Lat",
    "lng_num": "Lng",
    "steps_num": "steps_sum",
})
if park_class_col is not None and park_class_col in park_agg.columns:
    park_agg = park_agg.rename(columns={park_class_col: "park_class"})
if park_class_name_col is not None and park_class_name_col in park_agg.columns:
    park_agg = park_agg.rename(columns={park_class_name_col: "park_class_name"})

if "park_class" not in park_agg.columns and "park_class_name" in park_agg.columns:
    name_map = {
        "City block park": "A",
        "Neighborhood Park": "B",
        "District Park": "C",
        "Comprehensive Park": "D",
        "Regional Park": "E",
    }
    park_agg["park_class"] = park_agg["park_class_name"].map(name_map)

park_agg["annual_benefit_yen_2019"] = (
    park_agg["steps_sum"].fillna(0.0) * YEN_PER_STEP_2019 * EXPANSION_FACTOR
)

cost = pd.read_excel(COST_XLSX)

cost_osm_col = pick_column(
    cost.columns,
    exact_candidates=("osm_id",),
    contains_any=("osm_id",),
    label="cost osm_id"
)
cost["osm_id_norm"] = norm_osm(cost[cost_osm_col])

land_price_col = pick_column(
    cost.columns,
    exact_candidates=("price_fina", "land_price", "price", "unit_land_price"),
    contains_any=("price_fina", "land_price", "price"),
    label="land price"
)

om_unit_col = pick_column(
    cost.columns,
    exact_candidates=("unit_maintenance_yen_per_m2_2024", "maint", "maintenance", "unit_maintenance"),
    contains_any=("maint", "maintenance"),
    label="maintenance unit cost"
)

cost["land_price_num"] = pd.to_numeric(cost[land_price_col], errors="coerce")
cost["om_unit_raw"] = pd.to_numeric(cost[om_unit_col], errors="coerce")

cost_use = cost[["osm_id_norm", "land_price_num", "om_unit_raw"]].copy()

df = park_agg.merge(cost_use, on="osm_id_norm", how="left")

df["maint_2019"] = df["om_unit_raw"] * OM_TO_2019_FACTOR

df["construction_unit_cost_2019"] = CONSTRUCTION_UNIT_COST_2014 * CONSTRUCTION_TO_2019_FACTOR

df["area_m2"] = pd.to_numeric(df["area_m2"], errors="coerce")

df["land_cost_2019"] = df["land_price_num"] * df["area_m2"]
df["construction_cost_2019"] = df["construction_unit_cost_2019"] * df["area_m2"]
df["one_time_investment_2019"] = df["land_cost_2019"] + df["construction_cost_2019"]

df["annual_om_cost_2019"] = df["maint_2019"] * df["area_m2"]

df["annual_net_benefit_2019"] = df["annual_benefit_yen_2019"] - df["annual_om_cost_2019"]

results = df.apply(
    lambda row: discounted_payback_with_growth(
        invest=row["one_time_investment_2019"],
        annual_benefit_0=row["annual_benefit_yen_2019"],
        annual_om_0=row["annual_om_cost_2019"],
        r=DISCOUNT_RATE,
        g_b=BENEFIT_GROWTH,
        g_om=OM_GROWTH,
        years=EVAL_YEARS
    ),
    axis=1,
    result_type="expand"
)

results.columns = [
    "discounted_payback_years_full_growth",
    "discounted_payback_years_50y_growth",
    "npv_50y_growth",
    "discounted_payback_status_50y_growth"
]

df = pd.concat([df, results], axis=1)

df["payback_group_growth"] = df["discounted_payback_years_full_growth"].apply(classify_payback_year)

if "park_class" not in df.columns:
    raise KeyError("park_class is required for the A-E category summary; check park_class and park_class_name in the visit tables.")

type_summary = (
    df.groupby("park_class", dropna=False)
      .agg(
          parks_n=("osm_id_norm", "count"),
          annual_benefit_sum_2019=("annual_benefit_yen_2019", "sum"),
          annual_om_sum_2019=("annual_om_cost_2019", "sum"),
          investment_sum_2019=("one_time_investment_2019", "sum"),
          npv_50y_sum_growth=("npv_50y_growth", "sum"),
      )
      .reset_index()
)

group_dist = (
    df.groupby(["park_class", "payback_group_growth"], dropna=False)
      .size()
      .reset_index(name="n")
)

group_dist["pct"] = group_dist.groupby("park_class")["n"].transform(lambda s: s / s.sum() * 100)

type_dist_wide = (
    group_dist.pivot(index="park_class", columns="payback_group_growth", values="pct")
             .fillna(0.0)
             .reset_index()
)

for c in ["0-5", "5-20", "20-50", ">50", "never", "invalid"]:
    if c not in type_dist_wide.columns:
        type_dist_wide[c] = 0.0

type_dist_wide = type_dist_wide[
    ["park_class", "0-5", "5-20", "20-50", ">50", "never", "invalid"]
].copy()

type_summary = type_summary.merge(type_dist_wide, on="park_class", how="left")

if "park_class_name" in df.columns:
    class_name_map = (
        df.groupby("park_class", dropna=False)["park_class_name"]
          .agg(first_valid)
          .reset_index()
    )
    type_summary = class_name_map.merge(type_summary, on="park_class", how="right")

df.to_csv(OUT_TABLE_CSV, index=False, encoding="utf-8-sig")
type_summary.to_csv(OUT_TYPE_SUMMARY_CSV, index=False, encoding="utf-8-sig")

print("\n================ Model parameters ================")
print(f"BASE_YEAR         = {BASE_YEAR}")
print(f"EVAL_YEARS        = {EVAL_YEARS}")
print(f"DISCOUNT_RATE     = {DISCOUNT_RATE:.2%}")
print(f"BENEFIT_GROWTH    = {BENEFIT_GROWTH:.2%}")
print(f"OM_GROWTH         = {OM_GROWTH:.2%}")
print(f"EXPANSION_FACTOR  = {EXPANSION_FACTOR}")
print(f"YEN_PER_STEP_2019 = {YEN_PER_STEP_2019}")

print("\n================ Overall results ================")
overall_dist = df["payback_group_growth"].value_counts(dropna=False)
overall_pct = df["payback_group_growth"].value_counts(normalize=True, dropna=False) * 100
overall_show = pd.DataFrame({
    "n": overall_dist,
    "pct": overall_pct.round(2)
})
print(overall_show)

print("\n================ Results by park type ================")
show_cols = ["park_class"]
if "park_class_name" in type_summary.columns:
    show_cols.append("park_class_name")
show_cols += ["parks_n", "0-5", "5-20", "20-50", ">50", "never"]
print(type_summary[show_cols].round(2))

with open(OUT_DIAG_TXT, "w", encoding="utf-8") as f:
    f.write("=== SETTINGS ===\n")
    f.write(f"BASE_YEAR = {BASE_YEAR}\n")
    f.write(f"EVAL_YEARS = {EVAL_YEARS}\n")
    f.write(f"DISCOUNT_RATE = {DISCOUNT_RATE:.6f}\n")
    f.write(f"BENEFIT_GROWTH = {BENEFIT_GROWTH:.6f}\n")
    f.write(f"OM_GROWTH = {OM_GROWTH:.6f}\n")
    f.write(f"EXPANSION_FACTOR = {EXPANSION_FACTOR}\n")
    f.write(f"YEN_PER_STEP_2019 = {YEN_PER_STEP_2019:.8f}\n\n")

    f.write("=== OVERALL DISTRIBUTION ===\n")
    f.write(overall_show.to_string())
    f.write("\n\n=== TYPE SUMMARY ===\n")
    f.write(type_summary.round(4).to_string(index=False))

print("\nOutputs written:")
print(OUT_TABLE_CSV)
print(OUT_TYPE_SUMMARY_CSV)
print(OUT_DIAG_TXT)

print(f"\nDone in {time.perf_counter() - t0:.1f}s")


In [ ]:
# Evaluate payback sensitivity to discount and annual growth assumptions.
DISCOUNT_RATE_LIST = [0.01, 0.02, 0.04]
BENEFIT_GROWTH_LIST = [0.00, 0.01, 0.02, 0.03]
OM_GROWTH_LIST = [0.00, 0.01, 0.02, 0.03]

SENS_OUT_SUMMARY = os.path.join(
    OUT_DIR,
    "park_payback_sensitivity_summary_r_gb_gom.csv"
)
SENS_OUT_TYPE = os.path.join(
    OUT_DIR,
    "park_payback_sensitivity_by_type_r_gb_gom.csv"
)

sens_rows = []
sens_type_rows = []

base_df = df.copy()

# The sensitivity grid changes one accounting assumption combination at a time.
for r in DISCOUNT_RATE_LIST:
    for g_b in BENEFIT_GROWTH_LIST:
        for g_om in OM_GROWTH_LIST:
            tmp = base_df.copy()

            res = tmp.apply(
                lambda row: discounted_payback_with_growth(
                    invest=row["one_time_investment_2019"],
                    annual_benefit_0=row["annual_benefit_yen_2019"],
                    annual_om_0=row["annual_om_cost_2019"],
                    r=r,
                    g_b=g_b,
                    g_om=g_om,
                    years=EVAL_YEARS
                ),
                axis=1,
                result_type="expand"
            )

            res.columns = [
                "pb_full",
                "pb_50y",
                "npv_50y",
                "status_50y"
            ]
            tmp = pd.concat([tmp, res], axis=1)
            tmp["payback_group"] = tmp["pb_full"].apply(classify_payback_year)

            sens_rows.append({
                "discount_rate": r,
                "benefit_growth": g_b,
                "om_growth": g_om,
                "parks_n": len(tmp),
                "recoup_share_50y": np.isfinite(tmp["pb_50y"]).mean() * 100,
                "npv_positive_share_50y": (tmp["npv_50y"] > 0).mean() * 100,
                "never_payback_share": np.isinf(tmp["pb_full"]).mean() * 100,
                "share_0_5": (tmp["payback_group"] == "0-5").mean() * 100,
                "share_5_20": (tmp["payback_group"] == "5-20").mean() * 100,
                "share_20_50": (tmp["payback_group"] == "20-50").mean() * 100,
                "share_gt_50": (tmp["payback_group"] == ">50").mean() * 100,
                "share_never": (tmp["payback_group"] == "never").mean() * 100,
                "median_pb_50y": tmp["pb_50y"].median(skipna=True),
                "mean_pb_50y": tmp["pb_50y"].mean(skipna=True),
            })

            type_dist = (
                tmp.groupby(["park_class", "payback_group"], dropna=False)
                   .size()
                   .reset_index(name="n")
            )
            type_dist["pct"] = type_dist.groupby("park_class")["n"].transform(lambda s: s / s.sum() * 100)

            type_wide = (
                type_dist.pivot(index="park_class", columns="payback_group", values="pct")
                         .fillna(0.0)
                         .reset_index()
            )

            for c in ["0-5", "5-20", "20-50", ">50", "never", "invalid"]:
                if c not in type_wide.columns:
                    type_wide[c] = 0.0

            type_wide["discount_rate"] = r
            type_wide["benefit_growth"] = g_b
            type_wide["om_growth"] = g_om

            sens_type_rows.append(type_wide)

sens_summary = pd.DataFrame(sens_rows)
sens_type_summary = pd.concat(sens_type_rows, ignore_index=True)

sens_summary.to_csv(SENS_OUT_SUMMARY, index=False, encoding="utf-8-sig")
sens_type_summary.to_csv(SENS_OUT_TYPE, index=False, encoding="utf-8-sig")

print("\n================ Sensitivity analysis completed ================")
print("Overall summary:", SENS_OUT_SUMMARY)
print("Park-type summary:", SENS_OUT_TYPE)

print("\n---- Discount-rate sensitivity (g_b = 2%, g_om = 2%) ----")
print(
    sens_summary[
        (sens_summary["benefit_growth"] == 0.02) &
        (sens_summary["om_growth"] == 0.02)
    ][[
        "discount_rate", "recoup_share_50y", "npv_positive_share_50y",
        "never_payback_share", "share_0_5", "share_5_20", "share_20_50"
    ]].round(2)
)

print("\n---- Growth-rate sensitivity (r = 2%) ----")
print(
    sens_summary[
        sens_summary["discount_rate"] == 0.02
    ][[
        "benefit_growth", "om_growth", "recoup_share_50y",
        "npv_positive_share_50y", "never_payback_share"
    ]].round(2).sort_values(["benefit_growth", "om_growth"])
)


In [ ]:
# Map park-level discounted payback categories and investment scale.
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["axes.unicode_minus"] = False

PARKS_CSV = r"outputs/park_payback_growth_discounted_2019_50y_r2_b1_om1.csv"
BASE_SHP = r"data/public/urban_parks/urban_parks_study_area.shp"

SAVE_FIG = False
OUT_FIG = os.path.join("outputs", "panel_a_map_donuts_updated.png")

PAYBACK_COL = "discounted_payback_years_full_growth"

SIZE_Q = 0.90
SIZE_MIN = 4
SIZE_MAX = 110

FS_PANEL = 22
FS_PIE_PCT = 11
FS_TYPE_TITLE = 14.5
FS_LEGEND = 13.8

COLOR_RED = "#cc3333"
COLOR_GREEN = "#2f9a50"
COLOR_BLUE = "#3257b7"
COLOR_LIGHTGRAY = "#d3d3d3"
COLOR_DARKGRAY = "#8f8f8f"
BG_COLOR = "white"

CAT_ORDER = ["0-5", "5-20", "20-50", ">50 finite", "Never"]
CAT_COLORS = [COLOR_RED, COLOR_GREEN, COLOR_BLUE, COLOR_LIGHTGRAY, COLOR_DARKGRAY]

TYPE_ORDER = ["A", "B", "C", "D", "E"]
TYPE_NAME_MAP = {
    "A": "Block Park",
    "B": "Neighborhood Park",
    "C": "District Park",
    "D": "Comprehensive Park",
    "E": "Regional Park"
}

def classify_payback_5(x):
    if pd.isna(x) or np.isinf(x):
        return "Never"
    if x <= 5:
        return "0-5"
    elif x <= 20:
        return "5-20"
    elif x <= 50:
        return "20-50"
    else:
        return ">50 finite"

def annotate_donut(ax, wedges, values, total):
    if total <= 0:
        return

    labels_info = []
    for w, v in zip(wedges, values):
        if v <= 0:
            continue
        pct = 100.0 * v / total
        label = f"{pct:.0f}%"
        ang = 0.5 * (w.theta1 + w.theta2)
        ang_rad = np.deg2rad(ang)
        x = np.cos(ang_rad)
        y = np.sin(ang_rad)
        labels_info.append({"pct": pct, "label": label, "x": x, "y": y})

    small_upper = [d for d in labels_info if d["pct"] < 10 and d["y"] > 0]
    small_upper = sorted(small_upper, key=lambda z: z["y"])

    for i, d in enumerate(small_upper):
        d["y_shift"] = (i - (len(small_upper) - 1) / 2.0) * 0.10

    for d in labels_info:
        d.setdefault("y_shift", 0.0)
        x, y, pct, label = d["x"], d["y"], d["pct"], d["label"]

        if pct < 10:
            r_line = 1.02
            r_text = 1.56
            ha = "left" if x >= 0 else "right"
            ax.annotate(
                label,
                xy=(r_line * x, r_line * y),
                xytext=(r_text * x, r_text * (y + d["y_shift"])),
                ha=ha, va="center", fontsize=FS_PIE_PCT,
                arrowprops=dict(
                    arrowstyle="-", color="0.35", lw=0.8,
                    shrinkA=0, shrinkB=0,
                    connectionstyle="arc3,rad=0.10"
                )
            )
        else:
            r_text = 1.15
            ax.text(r_text * x, r_text * y, label,
                    ha="center", va="center", fontsize=FS_PIE_PCT)

df = pd.read_csv(PARKS_CSV, encoding="utf-8-sig").copy()
print("Loaded:", PARKS_CSV)

for c in ["Lng", "Lat", "area_m2", PAYBACK_COL]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

if "park_class" not in df.columns:
    raise KeyError("park_class column is required in PARKS_CSV")

df["payback_years"] = df[PAYBACK_COL]
df["payback_bin_5"] = df["payback_years"].apply(classify_payback_5)
df["map_group"] = df["payback_bin_5"]

print("\nOverall group counts:")
print(df["payback_bin_5"].value_counts(dropna=False))

print("\nBy park type (%):")
check = (
    df.groupby("park_class")["payback_bin_5"]
    .value_counts(normalize=True)
    .rename("share")
    .reset_index()
    .pivot(index="park_class", columns="payback_bin_5", values="share")
    .fillna(0)
)
print((check * 100).round(1))

df_map = df[np.isfinite(df["Lng"]) & np.isfinite(df["Lat"])].copy()

base = gpd.read_file(BASE_SHP)
base_outline = base.dissolve()

gdf = gpd.GeoDataFrame(
    df_map.copy(),
    geometry=gpd.points_from_xy(df_map["Lng"], df_map["Lat"]),
    crs="EPSG:4326"
)

if base_outline.crs is not None and str(base_outline.crs) != str(gdf.crs):
    gdf = gdf.to_crs(base_outline.crs)

area = pd.to_numeric(gdf["area_m2"], errors="coerce").fillna(0).clip(lower=0)
sqrt_area = np.sqrt(area)

denom = np.nanquantile(sqrt_area.replace(0, np.nan), SIZE_Q)
if not np.isfinite(denom) or denom <= 0:
    denom = np.nanmax(sqrt_area) if np.nanmax(sqrt_area) > 0 else 1.0

sizes = (sqrt_area / denom) * SIZE_MAX
sizes = sizes.clip(lower=SIZE_MIN, upper=SIZE_MAX)
sizes = sizes * 0.75
gdf["pt_size"] = sizes

g_0_5 = gdf[gdf["map_group"] == "0-5"].copy()
g_5_20 = gdf[gdf["map_group"] == "5-20"].copy()
g_20_50 = gdf[gdf["map_group"] == "20-50"].copy()
g_gt50 = gdf[gdf["map_group"] == ">50 finite"].copy()
g_never = gdf[gdf["map_group"] == "Never"].copy()

fig = plt.figure(figsize=(15.0, 6.1), facecolor=BG_COLOR)

gs_outer = fig.add_gridspec(
    nrows=1, ncols=2,
    width_ratios=[2.9, 6.0],
    left=0.02, right=0.992, top=0.965, bottom=0.11,
    wspace=0.035
)

gs_left = gs_outer[0, 0].subgridspec(
    nrows=2, ncols=3,
    hspace=0.78, wspace=0.62
)

ax_positions = [
    fig.add_subplot(gs_left[0, 0]),
    fig.add_subplot(gs_left[0, 1]),
    fig.add_subplot(gs_left[0, 2]),
    fig.add_subplot(gs_left[1, 0]),
    fig.add_subplot(gs_left[1, 1]),
]

for ax in ax_positions[:3]:
    pos = ax.get_position()
    ax.set_position([pos.x0, pos.y0 - 0.15, pos.width, pos.height])

ax_blank = fig.add_subplot(gs_left[1, 2])
ax_blank.axis("off")
ax_map = fig.add_subplot(gs_outer[0, 1])

for ax in ax_positions + [ax_map]:
    ax.set_facecolor(BG_COLOR)

for ax_pie, t in zip(ax_positions, TYPE_ORDER):
    sub = df[df["park_class"].astype(str) == t].copy()
    counts = sub["payback_bin_5"].value_counts().reindex(CAT_ORDER, fill_value=0)
    total = int(counts.sum())
    sizes_pie = counts.values if total > 0 else [0, 0, 0, 0, 1]

    wedges, _ = ax_pie.pie(
        sizes_pie,
        colors=CAT_COLORS,
        startangle=90,
        counterclock=False,
        radius=0.92,
        wedgeprops=dict(width=0.34, edgecolor=BG_COLOR)
    )
    annotate_donut(ax_pie, wedges, sizes_pie, total)

    ax_pie.set_aspect("equal")
    ax_pie.set_xticks([])
    ax_pie.set_yticks([])
    ax_pie.set_title(
        f"{t}. {TYPE_NAME_MAP[t]}",
        fontsize=FS_TYPE_TITLE,
        fontweight="bold",
        pad=30
    )

fig.text(0.012, 0.978, "a", fontsize=FS_PANEL, fontweight="bold", ha="left", va="top")

base_outline.boundary.plot(ax=ax_map, linewidth=1.05, color="0.6", zorder=1)

if len(g_never) > 0:
    ax_map.scatter(
        g_never.geometry.x, g_never.geometry.y,
        s=g_never["pt_size"].values,
        color=COLOR_DARKGRAY,
        alpha=0.62, linewidths=0, zorder=2
    )

if len(g_gt50) > 0:
    ax_map.scatter(
        g_gt50.geometry.x, g_gt50.geometry.y,
        s=g_gt50["pt_size"].values,
        color=COLOR_LIGHTGRAY,
        alpha=0.82, linewidths=0, zorder=2.15
    )

if len(g_20_50) > 0:
    ax_map.scatter(
        g_20_50.geometry.x, g_20_50.geometry.y,
        s=g_20_50["pt_size"].values,
        color=COLOR_BLUE,
        alpha=0.88, linewidths=0.08, edgecolors="white", zorder=3
    )

if len(g_5_20) > 0:
    ax_map.scatter(
        g_5_20.geometry.x, g_5_20.geometry.y,
        s=g_5_20["pt_size"].values,
        color=COLOR_GREEN,
        alpha=0.90, linewidths=0.08, edgecolors="white", zorder=3.2
    )

if len(g_0_5) > 0:
    ax_map.scatter(
        g_0_5.geometry.x, g_0_5.geometry.y,
        s=g_0_5["pt_size"].values,
        color=COLOR_RED,
        alpha=0.93, linewidths=0.08, edgecolors="white", zorder=3.4
    )

ax_map.set_axis_off()

xmin, ymin, xmax, ymax = base_outline.total_bounds
xpad = (xmax - xmin) * 0.01
ypad = (ymax - ymin) * 0.01
ax_map.set_xlim(xmin - xpad, xmax + xpad)
ax_map.set_ylim(ymin - ypad, ymax + ypad)

handles_left = [
    Line2D([0], [0], color=COLOR_RED, lw=6, label="0-5"),
    Line2D([0], [0], color=COLOR_GREEN, lw=6, label="5-20"),
    Line2D([0], [0], color=COLOR_BLUE, lw=6, label="20-50"),
    Line2D([0], [0], color=COLOR_LIGHTGRAY, lw=6, label=">50 but finite"),
    Line2D([0], [0], color=COLOR_DARKGRAY, lw=6, label="Never"),
]

fig.legend(
    handles=handles_left,
    loc="lower left",
    bbox_to_anchor=(0.67, 0.015),
    ncol=5,
    frameon=False,
    fontsize=FS_LEGEND,
    handlelength=1.0,
    columnspacing=0.9,
    handletextpad=0.45
)

if SAVE_FIG:
    fig.savefig(OUT_FIG, dpi=300, bbox_inches="tight", facecolor="white")
    print("Saved:", OUT_FIG)

plt.show()


In [ ]:
# Compare payback distributions across economic scenarios and park types.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from scipy.special import erf

BASE_INPUT_CSV = r"outputs/ab_modeling_table_r2_rebuilt.csv"

SAVE_FIG = False
OUT_FIG = os.path.join("outputs", "panel_b_payback_sensitivity_smooth_ecdf.png")

MAIN_R = 0.02
MAIN_GB = 0.02
MAIN_GOM = 0.02

DISCOUNT_RATE_LIST = [0.01, 0.02, 0.04]
BENEFIT_GROWTH_LIST = [0.00, 0.01, 0.02, 0.03]
OM_GROWTH_LIST = [0.00, 0.01, 0.02, 0.03]

TYPE_ORDER = ["A", "B", "C", "D", "E"]
TYPE_NAMES = {
    "A": "A. Block Park",
    "B": "B. Neighborhood Park",
    "C": "C. District Park",
    "D": "D. Comprehensive Park",
    "E": "E. Regional Park",
}
TYPE_COLORS = {
    "A": "#1f77b4",
    "B": "#ff7f0e",
    "C": "#2ca02c",
    "D": "#d62728",
    "E": "#9467bd",
}

T1, T2, T3 = 5, 20, 50
X_MAX = 100
Y_MAX = 100
GRID_N = 1500

SMOOTH_BW_YEARS = 2.0

TEXT_SCALE = 1.85
MAX_YEAR_LONG = 500

plt.rcdefaults()
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["axes.unicode_minus"] = False

FONT_MAIN   = int(round(12 * TEXT_SCALE))
FONT_LABEL  = int(round(14 * TEXT_SCALE))
FONT_TICK   = int(round(12 * TEXT_SCALE))
FONT_LEGEND = int(round(11 * TEXT_SCALE))
FONT_ANNO   = int(round(11 * TEXT_SCALE))
FONT_PANEL  = int(round(16 * TEXT_SCALE))

plt.rcParams.update({
    "font.size": FONT_MAIN,
    "axes.labelsize": FONT_LABEL,
    "xtick.labelsize": FONT_TICK,
    "ytick.labelsize": FONT_TICK,
    "legend.fontsize": FONT_LEGEND,
})

def discounted_payback_with_growth(invest, annual_benefit_0, annual_om_0,
                                   r, g_b, g_om, max_year_long=500):
    """
    Return full discounted payback year.
    If never pays back within max_year_long, return np.inf.
    """
    if not np.isfinite(invest) or invest <= 0:
        return np.nan
    if not np.isfinite(annual_benefit_0) or annual_benefit_0 <= 0:
        return np.inf
    if not np.isfinite(annual_om_0) or annual_om_0 < 0:
        return np.inf

    cum_pv = 0.0
    for t in range(1, max_year_long + 1):
        benefit_t = annual_benefit_0 * ((1 + g_b) ** (t - 1))
        om_t = annual_om_0 * ((1 + g_om) ** (t - 1))
        net_t = benefit_t - om_t
        pv_t = net_t / ((1 + r) ** t)
        cum_pv += pv_t
        if cum_pv >= invest:
            return float(t)
    return np.inf

def norm_cdf(z):
    return 0.5 * (1.0 + erf(z / np.sqrt(2.0)))

def smooth_ecdf_curve(Tvals_all, x_grid, bw=2.0):
    """
    Smoothed ECDF using Gaussian kernel.
    Important:
    - denominator = all parks in this type
    - finite positive paybacks contribute smoothly
    - inf / nan / never remain in denominator but are never accumulated
    """
    Tvals_all = np.asarray(Tvals_all, dtype=float)
    n_total = len(Tvals_all)

    if n_total == 0:
        return np.full_like(x_grid, np.nan, dtype=float)

    finite = np.isfinite(Tvals_all) & (Tvals_all > 0)
    T = Tvals_all[finite]

    if len(T) == 0:
        return np.zeros_like(x_grid, dtype=float)

    z = (x_grid[:, None] - T[None, :]) / bw
    y = 100.0 * norm_cdf(z).sum(axis=1) / n_total
    y = np.clip(y, 0, 100)

    y = np.maximum.accumulate(y)
    return y

df = pd.read_csv(BASE_INPUT_CSV, encoding="utf-8-sig", low_memory=False).copy()

need = [
    "park_class",
    "one_time_investment_2019",
    "annual_benefit_yen_2019",
    "annual_om_cost_2019"
]
miss = [c for c in need if c not in df.columns]
if miss:
    raise KeyError(f"Missing required columns in base file: {miss}")

df["park_class"] = df["park_class"].astype(str).str.strip().str.upper()
for c in ["one_time_investment_2019", "annual_benefit_yen_2019", "annual_om_cost_2019"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("Loaded:", BASE_INPUT_CSV)
print("Shape:", df.shape)
print("\nCounts by type:")
print(df["park_class"].value_counts(dropna=False).sort_index())

scenario_curves = {t: [] for t in TYPE_ORDER}
central_curves = {t: None for t in TYPE_ORDER}

x_grid = np.linspace(0, X_MAX, GRID_N)

for r in DISCOUNT_RATE_LIST:
    for g_b in BENEFIT_GROWTH_LIST:
        for g_om in OM_GROWTH_LIST:
            tmp = df.copy()

            tmp["payback_years"] = tmp.apply(
                lambda row: discounted_payback_with_growth(
                    invest=row["one_time_investment_2019"],
                    annual_benefit_0=row["annual_benefit_yen_2019"],
                    annual_om_0=row["annual_om_cost_2019"],
                    r=r,
                    g_b=g_b,
                    g_om=g_om,
                    max_year_long=MAX_YEAR_LONG
                ),
                axis=1
            )

            for t in TYPE_ORDER:
                sub = tmp[tmp["park_class"] == t].copy()
                y = smooth_ecdf_curve(sub["payback_years"].to_numpy(dtype=float), x_grid, bw=SMOOTH_BW_YEARS)
                scenario_curves[t].append(y)

                if np.isclose(r, MAIN_R) and np.isclose(g_b, MAIN_GB) and np.isclose(g_om, MAIN_GOM):
                    central_curves[t] = y.copy()

lower_curves = {}
upper_curves = {}

for t in TYPE_ORDER:
    if len(scenario_curves[t]) == 0:
        lower_curves[t] = np.full_like(x_grid, np.nan, dtype=float)
        upper_curves[t] = np.full_like(x_grid, np.nan, dtype=float)
        continue

    stack = np.vstack(scenario_curves[t])
    lower_curves[t] = np.nanmin(stack, axis=0)
    upper_curves[t] = np.nanmax(stack, axis=0)

print("\nMain scenario cumulative shares at key horizons:")
for t in TYPE_ORDER:
    sub_main = df[df["park_class"] == t].copy()
    if len(sub_main) == 0:
        continue

    sub_main["payback_years_main"] = sub_main.apply(
        lambda row: discounted_payback_with_growth(
            invest=row["one_time_investment_2019"],
            annual_benefit_0=row["annual_benefit_yen_2019"],
            annual_om_0=row["annual_om_cost_2019"],
            r=MAIN_R, g_b=MAIN_GB, g_om=MAIN_GOM,
            max_year_long=MAX_YEAR_LONG
        ),
        axis=1
    )

    T = sub_main["payback_years_main"].to_numpy(dtype=float)
    n = len(T)
    share_5 = 100.0 * np.mean(np.isfinite(T) & (T > 0) & (T <= 5))
    share_20 = 100.0 * np.mean(np.isfinite(T) & (T > 0) & (T <= 20))
    share_50 = 100.0 * np.mean(np.isfinite(T) & (T > 0) & (T <= 50))
    print(f"{t}: <=5y={share_5:.2f}%, <=20y={share_20:.2f}%, <=50y={share_50:.2f}% (n={n})")

fig, ax = plt.subplots(figsize=(14, 10), facecolor="white")
ax.set_facecolor("white")

ax.axvspan(0,  T1, alpha=0.08, color="steelblue")
ax.axvspan(T1, T2, alpha=0.06, color="steelblue")
ax.axvspan(T2, T3, alpha=0.05, color="steelblue")

legend_handles = []
legend_labels = []

for t in TYPE_ORDER:
    c = TYPE_COLORS[t]

    y_low = lower_curves[t]
    y_up = upper_curves[t]
    y_mid = central_curves[t]

    if y_mid is None or np.all(np.isnan(y_mid)):
        continue

    ax.fill_between(
        x_grid, y_low, y_up,
        color=c, alpha=0.20,
        linewidth=0, zorder=2
    )

    line, = ax.plot(
        x_grid, y_mid,
        color=c, linewidth=2.6, zorder=3
    )

    legend_handles.append(line)
    legend_labels.append(TYPE_NAMES[t])

ax.axhline(50, ls="--", lw=1.2, alpha=0.45, color="dodgerblue")
for tline, lab, ypos in [(T1, "5y", 30), (T2, "20y", 30), (T3, "50y", 52)]:
    ax.axvline(tline, ls="--", lw=1.2, alpha=0.60, color="dodgerblue")
    ax.text(
        tline, ypos, lab,
        ha="center", va="bottom",
        alpha=0.78, fontsize=FONT_ANNO
    )

ax.set_xlabel("Payback time (years)")
ax.set_ylabel("Share of parks with payback ≤ t (%)", labelpad=18)
ax.yaxis.set_label_coords(-0.09, 0.5)

ax.set_xlim(0, X_MAX)
ax.set_ylim(0, Y_MAX)
ax.grid(True, alpha=0.20)

leg1 = ax.legend(
    handles=legend_handles,
    labels=legend_labels,
    loc="upper left",
    ncol=1,
    frameon=True,
    framealpha=0.92,
    borderpad=0.6,
    handlelength=2.8
)

band_patch = Patch(
    facecolor="gray", alpha=0.22, edgecolor="none",
    label=f"Sensitivity envelope"
)
mid_line = Line2D(
    [0], [0], color="black", lw=2.5,
    label=f"Central curve (r=2%, g=2%)"
)

leg2 = ax.legend(
    handles=[band_patch, mid_line],
    loc="upper right",
    bbox_to_anchor=(0.985, 0.985),
    borderaxespad=0.2,
    frameon=True,
    framealpha=0.92
)
ax.add_artist(leg1)

fig.text(0.015, 0.985, "b", fontsize=FONT_PANEL, fontweight="bold", ha="left", va="top")

plt.tight_layout()

if SAVE_FIG:
    plt.savefig(OUT_FIG, dpi=300, bbox_inches="tight", facecolor="white")
    print("Saved:", OUT_FIG)

plt.show()


In [ ]:
# Plot annualized discounted margins by park type.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SOURCE_DF = pd.read_csv(
    r"outputs/park_payback_growth_discounted_2019_50y_r2_b1_om1.csv",
    encoding="utf-8-sig"
).copy()

BIN_W = 2000.0
TEXT_SCALE = 1.55

MAIN_RATE = 0.02
EVAL_YEARS = 50

SAVE_FIG = False
OUT_FIG = os.path.join("outputs", "panel_c_discounted_annual_margin_per_m2_updated.png")

TYPE_NAME = {
    "A": "A. Block Park",
    "B": "B. Neighborhood Park",
    "C": "C. District Park",
    "D": "D. Comprehensive Park",
    "E": "E. Regional Park",
}

print("Loaded CSV: main scenario growth-discount results")
print("MAIN_RATE:", MAIN_RATE)

plt.rcdefaults()
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["axes.unicode_minus"] = False

BASE_FONT   = int(round(10 * TEXT_SCALE))
LABEL_FONT  = int(round(10 * TEXT_SCALE))
TICK_FONT   = int(round(10 * TEXT_SCALE))
LEGEND_FONT = int(round(8.0 * TEXT_SCALE))
PANEL_FONT  = int(round(12 * TEXT_SCALE))
LINE_NOTE_FONT = int(round(8.8 * TEXT_SCALE))
LINE_VALUE_FONT = int(round(8.8 * TEXT_SCALE))

plt.rcParams.update({
    "font.size": BASE_FONT,
    "axes.labelsize": LABEL_FONT,
    "xtick.labelsize": TICK_FONT,
    "ytick.labelsize": TICK_FONT,
    "legend.fontsize": LEGEND_FONT,
})

need_cols = [
    "park_class",
    "area_m2",
    "npv_50y_growth",
]
miss = [c for c in need_cols if c not in SOURCE_DF.columns]
if miss:
    raise KeyError(f"Missing columns: {miss}. The main-scenario park-level result table is required.")

d = SOURCE_DF.copy()

d["park_class"] = d["park_class"].astype(str).str.strip()
d["area_m2"] = pd.to_numeric(d["area_m2"], errors="coerce")
d["npv_50y_growth"] = pd.to_numeric(d["npv_50y_growth"], errors="coerce")

d = d[d["park_class"].isin(list("ABCDE"))].copy()
d = d[np.isfinite(d["area_m2"]) & (d["area_m2"] > 0)].copy()
d = d[np.isfinite(d["npv_50y_growth"])].copy()

if len(d) == 0:
    raise ValueError("No valid records are available for plotting.")

print("Rows used:", len(d))

pv_factor_50y = np.sum(1.0 / (1.0 + MAIN_RATE) ** np.arange(1, EVAL_YEARS + 1))
print("PV factor (50y):", pv_factor_50y)

d["economic_margin_yen_per_m2"] = (
    d["npv_50y_growth"] / pv_factor_50y
) / d["area_m2"]

x_all = pd.to_numeric(d["economic_margin_yen_per_m2"], errors="coerce")
x_all = x_all[np.isfinite(x_all)]

if x_all.size == 0:
    raise ValueError("economic_margin_yen_per_m2 has no valid values.")

q05, q95 = np.percentile(x_all.values.astype(float), [5, 95])

inrange = (d["economic_margin_yen_per_m2"] >= q05) & (d["economic_margin_yen_per_m2"] <= q95)
d_in = d[inrange].copy()
excluded = len(d) - len(d_in)

x = d_in["economic_margin_yen_per_m2"].values.astype(float)
mean_x = float(np.mean(x))
median_x = float(np.median(x))

print(f"5%-95% range: [{q05:.2f}, {q95:.2f}]")
print(f"N shown: {len(d_in):,}")
print(f"N excluded: {excluded:,}")
print(f"Mean: {mean_x:.2f}")
print(f"Median: {median_x:.2f}")

left_data = np.floor(x.min() / BIN_W) * BIN_W
right_data = np.ceil(x.max() / BIN_W) * BIN_W

left = min(left_data, -10000)
right = max(right_data, 15000)

bins = np.arange(left, right + BIN_W, BIN_W)

groups = ["A", "B", "C", "D", "E"]
data_by_g = [
    d_in.loc[d_in["park_class"] == g, "economic_margin_yen_per_m2"].values.astype(float)
    for g in groups
]
legend_labels = [TYPE_NAME[g] for g in groups]

print(f"Histogram x-range: [{left:.0f}, {right:.0f}]")
print(f"Bin width = {BIN_W:.0f}, number of bins = {len(bins)-1}")

fig, ax = plt.subplots(figsize=(7.8, 5.6), facecolor="white")
ax.set_facecolor("white")

ax.hist(
    data_by_g,
    bins=bins,
    stacked=True,
    rwidth=0.82,
    edgecolor="white",
    linewidth=0.45,
    label=legend_labels,
)

ax.axvline(0, linestyle="--", linewidth=1.0, color="0.4")

ax.axvline(mean_x, linestyle="--", linewidth=1.45, color="red", alpha=0.95)
ax.axvline(median_x, linestyle="--", linewidth=1.45, color="black", alpha=0.95)

ax.grid(True, alpha=0.20)

ax.set_xlabel("Discounted annual economic margin per unit area (yen/m²·year)")
ax.set_ylabel("Number of parks")

ax.text(
    0.01, 1.02, "c",
    transform=ax.transAxes,
    ha="left", va="bottom",
    fontsize=PANEL_FONT, fontweight="bold"
)

leg = ax.legend(
    loc="upper right",
    bbox_to_anchor=(0.985, 0.985),
    borderaxespad=0.2,
    frameon=True,
    framealpha=0.82,
    handlelength=1.4,
    labelspacing=0.35,
    borderpad=0.35,
)
for t in leg.get_texts():
    t.set_fontsize(t.get_fontsize() * 0.92)

ax.set_xlim(left, right)

tick_start = int(np.ceil(left / 5000.0) * 5000)
tick_end = int(np.floor(right / 5000.0) * 5000)
xticks = np.arange(tick_start, tick_end + 1, 5000)
if len(xticks) > 0:
    ax.set_xticks(xticks)

y_top = ax.get_ylim()[1]
x_span = right - left
x_offset = x_span * 0.014

close_lines = abs(mean_x - median_x) < x_span * 0.05

if close_lines:
    y_mean = y_top * 0.93
    y_median = y_top * 0.84
else:
    y_mean = y_top * 0.93
    y_median = y_top * 0.93

ax.text(
    mean_x + x_offset,
    y_mean,
    f"Mean = {mean_x:.0f}",
    ha="left",
    va="top",
    fontsize=LINE_VALUE_FONT,
    color="red",
    bbox=dict(facecolor="white", alpha=0.78, edgecolor="none", boxstyle="round,pad=0.18")
)

ax.text(
    median_x + x_offset,
    y_median,
    f"Median = {median_x:.0f}",
    ha="left",
    va="top",
    fontsize=LINE_VALUE_FONT,
    color="black",
    bbox=dict(facecolor="white", alpha=0.78, edgecolor="none", boxstyle="round,pad=0.18")
)

plt.tight_layout()

if SAVE_FIG:
    plt.savefig(OUT_FIG, dpi=300, bbox_inches="tight")
    print("Saved ->", OUT_FIG)

plt.show()


In [ ]:
# Assemble the park-level modeling table used in the planning-stage analysis.
import os
import time
import numpy as np
import pandas as pd

t0 = time.perf_counter()

VISIT_FILES = [
    r"data/restricted/mobility/mobility_part_0.csv",
    r"data/restricted/mobility/mobility_part_1.csv",
]

COST_XLSX = (
    r"data/restricted/park_costs/park_costs_2019_2024.xlsx"
)

EPOP_FILE = (
    r"outputs/park_exposure_Epop_alltypes.csv"
)

OUT_DIR = (
    r"outputs"
)
os.makedirs(OUT_DIR, exist_ok=True)

OUT_CSV = os.path.join(OUT_DIR, "ab_modeling_table_r2_rebuilt.csv")
OUT_XLSX = os.path.join(OUT_DIR, "ab_modeling_table_r2_rebuilt.xlsx")
OUT_DIAG_TXT = os.path.join(OUT_DIR, "ab_modeling_table_r2_rebuilt_diagnostics.txt")

BASE_YEAR = 2019
EVAL_YEARS = 50

DISCOUNT_RATE = 0.02
BENEFIT_GROWTH = 0.02
OM_GROWTH = 0.02

YEN_PER_STEP_2019 = 0.04056
EXPANSION_FACTOR = 150

CONSTRUCTION_UNIT_COST_2014 = 12000.0
AREA_IS_HECTARE_WHEN_ONLY_AREA = False

CPI_2003_2015BASE = 97.2
CPI_2014_2015BASE = 102.8 / 103.6 * 100.0
CPI_2019_2015BASE = 101.8

CONSTRUCTION_TO_2019_FACTOR = CPI_2019_2015BASE / CPI_2014_2015BASE
OM_TO_2019_FACTOR = CPI_2019_2015BASE / CPI_2003_2015BASE

def norm_osm(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\.0$", "", regex=True)
    s = s.replace({"nan": np.nan, "None": np.nan, "": np.nan})
    return s

def first_valid(series: pd.Series):
    s = series.dropna()
    return s.iloc[0] if len(s) else np.nan

def numeric_median(series: pd.Series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    return float(s.median()) if len(s) else np.nan

def pick_column(columns, exact_candidates=(), contains_all=(), contains_any=(),
                required=True, label="column"):
    cols = list(columns)
    lower_map = {c.lower(): c for c in cols}

    for cand in exact_candidates:
        if cand in cols:
            return cand
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]

    for c in cols:
        cl = c.lower()
        ok_all = all(k.lower() in cl for k in contains_all) if contains_all else True
        ok_any = any(k.lower() in cl for k in contains_any) if contains_any else True
        if ok_all and ok_any:
            return c

    if required:
        raise KeyError(f"Cannot find {label}. Available columns:\n{cols}")
    return None

def discounted_payback_with_growth(invest, annual_benefit_0, annual_om_0,
                                   r, g_b, g_om, years=50, max_year_long=500):
    """
    Return:
    - discounted_payback_years_r2
    - discounted_payback_years_50y_r2
    - npv_50y_r2
    - discounted_payback_status_full_r2
    - discounted_payback_status_50y_r2
    - is_finite_payback_r2
    - is_recoup_within_50y_r2
    """
    if not np.isfinite(invest) or not np.isfinite(annual_benefit_0) or not np.isfinite(annual_om_0):
        return np.nan, np.nan, np.nan, "invalid", "invalid", False, False

    if invest <= 0:
        npv_50y = 0.0
        for t in range(1, years + 1):
            benefit_t = annual_benefit_0 * ((1 + g_b) ** (t - 1))
            om_t = annual_om_0 * ((1 + g_om) ** (t - 1))
            net_t = benefit_t - om_t
            npv_50y += net_t / ((1 + r) ** t)
        return 0.0, 0.0, npv_50y, "finite payback", "recouped within 50 years", True, True

    cum_pv = 0.0
    payback_year_50 = np.nan
    npv_50y = -invest

    for t in range(1, years + 1):
        benefit_t = annual_benefit_0 * ((1 + g_b) ** (t - 1))
        om_t = annual_om_0 * ((1 + g_om) ** (t - 1))
        net_t = benefit_t - om_t
        pv_t = net_t / ((1 + r) ** t)

        cum_pv += pv_t
        npv_50y += pv_t

        if pd.isna(payback_year_50) and cum_pv >= invest:
            payback_year_50 = float(t)

    if pd.notna(payback_year_50):
        return payback_year_50, payback_year_50, npv_50y, "finite payback", "recouped within 50 years", True, True

    cum_pv_long = cum_pv
    payback_year_full = np.nan

    for t in range(years + 1, max_year_long + 1):
        benefit_t = annual_benefit_0 * ((1 + g_b) ** (t - 1))
        om_t = annual_om_0 * ((1 + g_om) ** (t - 1))
        net_t = benefit_t - om_t
        pv_t = net_t / ((1 + r) ** t)
        cum_pv_long += pv_t

        if pd.isna(payback_year_full) and cum_pv_long >= invest:
            payback_year_full = float(t)
            break

    if pd.notna(payback_year_full):
        return payback_year_full, np.nan, npv_50y, "finite payback", "not recouped within 50 years", True, False
    else:
        return np.inf, np.nan, npv_50y, "never pay back", "never pay back", False, False

def map_state_full(x):
    if pd.isna(x):
        return "invalid"
    if np.isinf(x):
        return "never"
    return "finite"

def map_state_50y(x):
    if pd.isna(x):
        return "beyond_50y"
    return "within_50y"

visit_frames = []
for fp in VISIT_FILES:
    df0 = pd.read_csv(fp, low_memory=False)
    visit_frames.append(df0)

visits = pd.concat(visit_frames, ignore_index=True)

osm_col = pick_column(
    visits.columns,
    exact_candidates=("osm_id",),
    contains_any=("osm_id",),
    label="visit osm_id"
)
steps_col = pick_column(
    visits.columns,
    exact_candidates=("steps",),
    contains_any=("steps",),
    label="steps"
)
lat_col = pick_column(
    visits.columns,
    exact_candidates=("Lat", "lat", "latitude"),
    contains_any=("lat",),
    label="lat"
)
lng_col = pick_column(
    visits.columns,
    exact_candidates=("Lng", "lng", "lon", "longitude"),
    contains_any=("lng", "lon"),
    label="lng"
)
area_col = pick_column(
    visits.columns,
    exact_candidates=("area_m2", "area"),
    contains_any=("area",),
    label="area"
)
park_class_col = pick_column(
    visits.columns,
    exact_candidates=("park_class",),
    contains_all=("park", "class"),
    contains_any=("class",),
    required=False,
    label="park_class"
)
park_class_name_col = pick_column(
    visits.columns,
    exact_candidates=("park_class_name",),
    contains_all=("park", "class", "name"),
    contains_any=("class_name", "park_class_name"),
    required=False,
    label="park_class_name"
)

visits = visits.copy()
visits["osm_id_norm"] = norm_osm(visits[osm_col])
visits["steps_num"] = pd.to_numeric(visits[steps_col], errors="coerce")
visits["lat_num"] = pd.to_numeric(visits[lat_col], errors="coerce")
visits["lng_num"] = pd.to_numeric(visits[lng_col], errors="coerce")
visits["area_raw"] = pd.to_numeric(visits[area_col], errors="coerce")

if AREA_IS_HECTARE_WHEN_ONLY_AREA:
    visits["area_m2"] = visits["area_raw"] * 10000.0
else:
    visits["area_m2"] = visits["area_raw"]

agg_dict = {
    "lat_num": "mean",
    "lng_num": "mean",
    "area_m2": numeric_median,
    "steps_num": "sum",
}
if park_class_col is not None:
    agg_dict[park_class_col] = first_valid
if park_class_name_col is not None:
    agg_dict[park_class_name_col] = first_valid

park_agg = (
    visits.loc[visits["osm_id_norm"].notna()]
    .groupby(["osm_id_norm"], dropna=False, as_index=False)
    .agg(agg_dict)
)

park_agg = park_agg.rename(columns={
    "lat_num": "Lat",
    "lng_num": "Lng",
    "steps_num": "steps_sum",
})
if park_class_col is not None and park_class_col in park_agg.columns:
    park_agg = park_agg.rename(columns={park_class_col: "park_class"})
if park_class_name_col is not None and park_class_name_col in park_agg.columns:
    park_agg = park_agg.rename(columns={park_class_name_col: "park_class_name"})

park_agg["annual_benefit_yen_2019"] = (
    park_agg["steps_sum"].fillna(0.0) * YEN_PER_STEP_2019 * EXPANSION_FACTOR
)

cost = pd.read_excel(COST_XLSX)

cost_osm_col = pick_column(
    cost.columns,
    exact_candidates=("osm_id",),
    contains_any=("osm_id",),
    label="cost osm_id"
)
land_price_col = pick_column(
    cost.columns,
    exact_candidates=("price_fina", "land_price", "price", "unit_land_price"),
    contains_any=("price_fina", "land_price", "price"),
    label="land price"
)
om_unit_col = pick_column(
    cost.columns,
    exact_candidates=("unit_maintenance_yen_per_m2_2024", "maint", "maintenance", "unit_maintenance"),
    contains_any=("maint", "maintenance"),
    label="maintenance unit cost"
)

cost = cost.copy()
cost["osm_id_norm"] = norm_osm(cost[cost_osm_col])
cost["land_price"] = pd.to_numeric(cost[land_price_col], errors="coerce")
cost["om_unit_raw"] = pd.to_numeric(cost[om_unit_col], errors="coerce")

cost_use = cost[["osm_id_norm", "land_price", "om_unit_raw"]].copy()

epop = pd.read_csv(EPOP_FILE, low_memory=False)

epop_osm_col = pick_column(
    epop.columns,
    exact_candidates=("osm_id_norm", "osm_id"),
    contains_any=("osm_id_norm", "osm_id"),
    label="E_pop osm_id"
)
epop_col = pick_column(
    epop.columns,
    exact_candidates=("E_pop",),
    contains_all=("e", "pop"),
    contains_any=("E_pop", "epop"),
    label="E_pop"
)
epop_per_m2_col = pick_column(
    epop.columns,
    exact_candidates=("E_pop_per_m2",),
    contains_all=("e", "pop", "m2"),
    contains_any=("E_pop_per_m2", "epop_per_m2"),
    required=False,
    label="E_pop_per_m2"
)

epop = epop.copy()
epop["osm_id_norm"] = norm_osm(epop[epop_osm_col])
epop["E_pop"] = pd.to_numeric(epop[epop_col], errors="coerce")

if epop_per_m2_col is not None:
    epop["E_pop_per_m2"] = pd.to_numeric(epop[epop_per_m2_col], errors="coerce")
else:
    epop["E_pop_per_m2"] = np.nan

epop_use = epop[["osm_id_norm", "E_pop", "E_pop_per_m2"]].copy()

epop_use = (
    epop_use.groupby("osm_id_norm", as_index=False)
    .agg({
        "E_pop": "median",
        "E_pop_per_m2": "median"
    })
)

df = park_agg.merge(cost_use, on="osm_id_norm", how="left")
df = df.merge(epop_use, on="osm_id_norm", how="left")

df["maint"] = df["om_unit_raw"] * OM_TO_2019_FACTOR

df["construction_unit_cost_2019"] = CONSTRUCTION_UNIT_COST_2014 * CONSTRUCTION_TO_2019_FACTOR

df["area_m2"] = pd.to_numeric(df["area_m2"], errors="coerce")

df["land_cost_2019"] = df["land_price"] * df["area_m2"]
df["construction_cost_2019"] = df["construction_unit_cost_2019"] * df["area_m2"]
df["one_time_investment_2019"] = df["land_cost_2019"] + df["construction_cost_2019"]

df["annual_om_cost_2019"] = df["maint"] * df["area_m2"]

df["annual_net_benefit_2019"] = df["annual_benefit_yen_2019"] - df["annual_om_cost_2019"]

mask_epop_per_m2_missing = df["E_pop_per_m2"].isna() & df["E_pop"].notna() & df["area_m2"].gt(0)
df.loc[mask_epop_per_m2_missing, "E_pop_per_m2"] = (
    df.loc[mask_epop_per_m2_missing, "E_pop"] / df.loc[mask_epop_per_m2_missing, "area_m2"]
)

results = df.apply(
    lambda row: discounted_payback_with_growth(
        invest=row["one_time_investment_2019"],
        annual_benefit_0=row["annual_benefit_yen_2019"],
        annual_om_0=row["annual_om_cost_2019"],
        r=DISCOUNT_RATE,
        g_b=BENEFIT_GROWTH,
        g_om=OM_GROWTH,
        years=EVAL_YEARS,
        max_year_long=500
    ),
    axis=1,
    result_type="expand"
)

results.columns = [
    "discounted_payback_years_r2",
    "discounted_payback_years_50y_r2",
    "npv_50y_r2",
    "discounted_payback_status_full_r2",
    "discounted_payback_status_50y_r2",
    "is_finite_payback_r2",
    "is_recoup_within_50y_r2",
]

df = pd.concat([df, results], axis=1)

df["payback_state_full_r2"] = df["discounted_payback_years_r2"].apply(map_state_full)
df["payback_state_50y_r2"] = df["discounted_payback_years_50y_r2"].apply(map_state_50y)

final_cols = [
    "osm_id_norm",
    "park_class",
    "park_class_name",
    "Lng",
    "Lat",
    "area_m2",
    "land_cost_2019",
    "land_price",
    "annual_om_cost_2019",
    "maint",
    "one_time_investment_2019",
    "annual_benefit_yen_2019",
    "annual_net_benefit_2019",
    "discounted_payback_years_r2",
    "payback_state_full_r2",
    "payback_state_50y_r2",
    "is_finite_payback_r2",
    "is_recoup_within_50y_r2",
    "E_pop",
    "E_pop_per_m2",
    "npv_50y_r2",
    "construction_cost_2019",
    "discounted_payback_years_50y_r2",
    "discounted_payback_status_full_r2",
    "discounted_payback_status_50y_r2",
]

for c in final_cols:
    if c not in df.columns:
        df[c] = np.nan

final_df = df[final_cols].copy()

final_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
final_df.to_excel(OUT_XLSX, index=False)

overall_dist = final_df["discounted_payback_status_full_r2"].value_counts(dropna=False)
overall_pct = final_df["discounted_payback_status_full_r2"].value_counts(normalize=True, dropna=False) * 100
overall_show = pd.DataFrame({
    "n": overall_dist,
    "pct": overall_pct.round(2)
})

with open(OUT_DIAG_TXT, "w", encoding="utf-8") as f:
    f.write("=== SETTINGS ===\n")
    f.write(f"BASE_YEAR = {BASE_YEAR}\n")
    f.write(f"EVAL_YEARS = {EVAL_YEARS}\n")
    f.write(f"DISCOUNT_RATE = {DISCOUNT_RATE:.6f}\n")
    f.write(f"BENEFIT_GROWTH = {BENEFIT_GROWTH:.6f}\n")
    f.write(f"OM_GROWTH = {OM_GROWTH:.6f}\n")
    f.write(f"EXPANSION_FACTOR = {EXPANSION_FACTOR}\n")
    f.write(f"YEN_PER_STEP_2019 = {YEN_PER_STEP_2019:.8f}\n\n")

    f.write("=== SOURCE FILES ===\n")
    f.write(f"VISIT_FILES = {VISIT_FILES}\n")
    f.write(f"COST_XLSX   = {COST_XLSX}\n")
    f.write(f"EPOP_FILE   = {EPOP_FILE}\n\n")

    f.write("=== SOURCE COLUMNS ===\n")
    f.write(f"VISIT osm_id          : {osm_col}\n")
    f.write(f"VISIT steps           : {steps_col}\n")
    f.write(f"VISIT Lat             : {lat_col}\n")
    f.write(f"VISIT Lng             : {lng_col}\n")
    f.write(f"VISIT area            : {area_col}\n")
    f.write(f"VISIT park_class      : {park_class_col}\n")
    f.write(f"VISIT park_class_name : {park_class_name_col}\n")
    f.write(f"COST osm_id           : {cost_osm_col}\n")
    f.write(f"COST land_price       : {land_price_col}\n")
    f.write(f"COST maint unit       : {om_unit_col}\n")
    f.write(f"EPOP osm_id           : {epop_osm_col}\n")
    f.write(f"EPOP E_pop            : {epop_col}\n")
    f.write(f"EPOP E_pop_per_m2     : {epop_per_m2_col}\n\n")

    f.write("=== OVERALL DISTRIBUTION ===\n")
    f.write(overall_show.to_string())
    f.write("\n")

print("\n================ Reconstruction completed ================")
print("CSV output:", OUT_CSV)
print("XLSX output:", OUT_XLSX)
print("Diagnostic output:", OUT_DIAG_TXT)
print("\nOverall payback-status distribution:")
print(overall_show)

print(f"\nDone in {time.perf_counter() - t0:.1f}s")
